<p class="lead">A notebook is an excellent laboratory bench and a poor product. The code in it runs once, in one order, on one machine, with results that live in the output of the chunks rather than in files. This unit performs the conversion that every long-lived piece of laboratory software goes through: the curation protocol of Unit 6 becomes an installable R package with modules, roxygen documentation, a configuration object, a command-line entry point, a `testthat` suite, continuous integration, a licence and a citation file. The package is written into a temporary directory by this document, documented with `roxygen2`, installed into a temporary library, imported, exercised, tested, broken on purpose, caught by the tests and finally deleted — so that everything claimed here is demonstrated rather than described.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad convierte el código de un cuaderno en software mantenible. Se refactoriza el protocolo de curación del curso en funciones puras con la entrada y la salida en los bordes, y se construye un paquete de R real (DESCRIPTION, NAMESPACE generado por <code>roxygen2</code>, módulos en <code>R/</code>, verificación de argumentos, configuración en YAML, registro de eventos y un punto de entrada para <code>Rscript</code>) dentro de un directorio temporal creado por el propio documento. El paquete se instala en una biblioteca temporal y se somete a una batería de pruebas con <code>testthat</code> —accesorios, casos tabulados, tolerancias numéricas, una prueba de regresión contra la tabla de referencia y propiedades como la idempotencia— ejecutada en vivo. Se inyecta a propósito un error de refactorización que supera la mayoría de las pruebas y sólo detectan la prueba escrita a mano y la de regresión. Finalmente se tratan el análisis estático (con un verificador escrito aquí, porque <code>lintr</code> no está instalado), los flujos de integración continua de GitHub Actions, el versionado semántico, el archivo NEWS.md, la licencia, <code>CITATION.cff</code> y una lista de verificación para revisar código ajeno.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Refactor notebook code into an R package (DESCRIPTION, NAMESPACE, modules in <code>R/</code>, roxygen documentation, an <code>Rscript</code> entry point) without changing the results — and prove it with tests.</li>
<li>Write unit and regression tests with <code>testthat</code> (fixtures, tabulated cases, tolerances for floats, golden files, property-style invariants), run them on an installed package and reason about coverage.</li>
<li>Use argument checking, logging, configuration files, static checks and semantic versioning.</li>
<li>Document a package in its four layers — README, roxygen help pages, vignette, reference site — and make it citable with NEWS.md, a licence and a CITATION.cff.</li>
<li>Review someone else's code constructively, and receive a review without taking it personally.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk in order; the package is created in a temporary folder, installed into a temporary library, and both are removed by the last chunk, so nothing is left behind. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","roxygen2","testthat","callr","yaml"))</code> once before the setup chunk. No network access and no cluster are needed. The package implements the curation protocol of <a href="../U6_Data_Quality/index.html">Unit 6</a>, whose reference output is shipped with the course and used here as the regression target. The whole document runs in about a minute, most of it spent in two package installations.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(ggplot2); library(patchwork)
  library(roxygen2);   library(testthat); library(callr); library(knitr)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
OKABE <- c("#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp  <- function(f) data_file(f)
TMP <- tempdir()                           ## scratch space; the last chunk removes everything
RAW_PATH <- normalizePath(dp("rock1_chembl_raw.csv"))      ## absolute: subprocesses need it
REF_PATH <- normalizePath(dp("rock1_chembl_curated.csv"))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| data.table", as.character(packageVersion("data.table")),
    "| roxygen2", as.character(packageVersion("roxygen2")),
    "| testthat", as.character(packageVersion("testthat")),
    "| callr", as.character(packageVersion("callr")), "\n")
cat("raw data :", basename(RAW_PATH), sprintf("(%.1f MB)", file.size(RAW_PATH) / 1e6), "\n")
cat("reference:", basename(REF_PATH), "\n")



## 1. Why the notebook has to end

Every thesis in the laboratory starts the same way: an R Markdown file that loads a table, cleans it, computes something and draws a figure. That is the correct way to start — the notebook is the default working instrument of data science for good reasons<sup class="c"><a href="#r15">[15]</a></sup>. The trouble appears six months later, when a second student needs the same cleaning for a different target, when a referee asks what exactly was excluded, or when the first student graduates and the notebook is the only description of a protocol that three papers depend on.

Notebooks fail as software for four specific reasons, all of them measurable. Their execution order is not the order of the chunks, so a notebook that runs correctly for its author may not run at all from top to bottom — a large-scale study of public notebooks found that only a small minority reproduced their own stored outputs<sup class="c"><a href="#r16">[16]</a></sup>. Their state is invisible: an object defined in a deleted chunk survives in the session and silently supports the rest of the analysis — R makes this worse than most languages, because a saved `.RData` restores that invisible state in the next session. They have no interface, so the only way to reuse the logic is to copy it, and copies diverge. And they cannot be tested, because there is nothing addressable to test.

The remedy is not to abandon notebooks — they remain the right medium for exploration, for teaching and for the narrative that accompanies a result<sup class="c"><a href="#r17">[17]</a></sup>. The remedy is to move the parts that other people depend on into a package, and to let the notebook `library()` it. The notebook then keeps the story and loses the responsibility. This is the organising idea behind every published set of rules for reproducible computational research<sup class="c"><a href="#r9">[9]</a><a href="#r4">[4]</a></sup>, and it is the practice that the reproducibility surveys of the last decade found missing<sup class="c"><a href="#r18">[18]</a><a href="#r10">[10]</a></sup>.

<div class="box bridge"><h4>Domain context — what the group's code is actually for</h4>
<p>Three pipelines in the laboratory outlive any single student: the <strong>curation</strong> of bioactivity data downloaded from ChEMBL into a modelling-ready table (the subject of this unit), the post-processing of <strong>virtual-screening</strong> campaigns into ranked lists with early-recognition metrics, and the analysis of <strong>molecular-dynamics</strong> trajectories into observables and their uncertainties. Each is used by several projects at once, and each produces numbers that end up in publications. When a paper carries a <em>software-availability statement</em> — "the code is available at …" — it commits the group to something concrete: the repository must exist, it must contain what produced the numbers, it must run on a machine that is not yours, and it must say what it does. Research software that cannot be rerun is a liability for the laboratory, not an asset<sup class="c"><a href="#r1">[1]</a><a href="#r3">[3]</a></sup>, and software errors have retracted real papers<sup class="c"><a href="#r24">[24]</a><a href="#r25">[25]</a></sup>, invalidated the statistics of a whole imaging literature<sup class="c"><a href="#r26">[26]</a></sup> and corrupted thousands of gene lists through a spreadsheet default<sup class="c"><a href="#r27">[27]</a></sup>. The data these pipelines consume come from public databases with their own release cycles<sup class="c"><a href="#r31">[31]</a></sup>, and they are processed with libraries that are themselves research software to be pinned and cited<sup class="c"><a href="#r29">[29]</a><a href="#r33">[33]</a></sup>.</p></div>

There is a second, newer reason to care. A large share of the code written in the group now begins as a suggestion from a language model. Such code is usually syntactically fine, frequently subtly wrong, and always written without knowledge of your data. Tests are what make it safe to accept: a suggestion that passes a regression test against a known-good output is a contribution, and one that does not is a bug caught in seconds rather than in review.



In [ ]:
raw <- fread(dp("rock1_chembl_raw.csv"), na.strings = c("", "NA"))
UNITS_TO_NM <- c(nM = 1, uM = 1e3, mM = 1e6, M = 1e9, pM = 1e-3, fM = 1e-6,
                 "nmol/L" = 1, "umol/L" = 1e3)
t0 <- Sys.time()
d <- raw[!is.na(canonical_smiles)]
d <- d[target_organism == "Homo sapiens"]
d <- d[is.na(data_validity_comment)]
d <- d[!is.na(standard_value)]
d[, value_nM := unname(UNITS_TO_NM[standard_units]) * standard_value]
d <- d[!is.na(value_nM) & value_nM >= 1e-12 & value_nM <= 1e8]
ref <- fread(dp("rock1_chembl_curated.csv"))                   ## stands in for RDKit, see the box below
map <- ref[, .(chembl_id = unlist(strsplit(chembl_ids, "|", fixed = TRUE))),
           by = .(inchikey, std_smiles, HeavyAtoms)]
d <- merge(d, map, by.x = "molecule_chembl_id", by.y = "chembl_id", all.x = TRUE, sort = FALSE)
d <- d[!is.na(inchikey) & HeavyAtoms >= 3 & HeavyAtoms <= 70]
d[, pActivity := 9 - log10(value_nM)]
d[, censored := !(is.na(standard_relation) | standard_relation == "=")]
notebook_table <- d[, {ex <- .SD[censored == FALSE]
                       use <- if (nrow(ex)) ex else .SD
                       .(pActivity = median(use$pActivity), censored = nrow(ex) == 0L,
                         n_measurements = .N)}, by = inchikey][order(inchikey)]
cat(sprintf("%d raw records -> %d usable measurements -> %d compounds in %.1f s\n",
            nrow(raw), nrow(d), nrow(notebook_table),
            as.numeric(difftime(Sys.time(), t0, units = "secs"))))
print(head(notebook_table, 3))



<div class="box bridge"><h4>Substitution — the standardisation step in R</h4>
<p>The Python edition standardises every structure with RDKit (normalise, largest fragment, uncharge) and keys the result by InChIKey. R has no RDKit binding in this environment, so the package built here cannot recompute structures. Instead it takes the standardisation as <em>data</em>: a <strong>structure-key table</strong> mapping each ChEMBL molecule identifier to the InChIKey, the standardised SMILES and the heavy-atom count, derived from the frozen curated file that RDKit produced (see <a href="../../data/manifest_fingerprints.json">manifest_fingerprints.json</a> and <code>data/rock1_curated.sdf.gz</code>, which ships the structures themselves). This is a substitution forced by the environment, and it is also a defensible design: the expensive, language-specific step becomes an injected dependency, which is exactly what makes the rest of the protocol testable without a chemistry toolkit. Everything else — the filters, the unit harmonisation, the activity transform, the aggregation rule — is implemented in R and reproduces the reference table exactly, as Section 2 shows.</p></div>

That chunk works. It is also unusable as software, and the reasons are worth naming one by one, because the refactoring in the next section is simply the removal of each.

1. It mixes input, computation and output: the file is read, the structures are joined and the table is produced in one block, so none of the three can be exercised without the others.
1. It communicates through session state: `d` is reassigned eight times, and the meaning of `d` depends on how many chunks above it have run.
1. Its parameters are buried: the heavy-atom window `3, 70`, the concentration bounds and the unit table are literals in the middle of expressions, invisible to anyone reading the Methods.
1. It has no interface: another project wanting the same protocol must copy the chunk, and the copy will not receive the fix made next month.
1. It cannot report what it did: the exclusion counts exist only as the difference between two `nrow()` calls that nobody printed.

## 2. Refactoring: pure functions with the input and output at the edges

The pattern that fixes all five is older than data science. Write **pure functions** — a function whose output depends only on its arguments and which changes nothing outside itself — for the computation, and keep reading and writing at the edges of the program<sup class="c"><a href="#r1">[1]</a><a href="#r2">[2]</a></sup>. A pure function is testable by construction: call it with an input, compare with the expected output, no fixtures, no files, no order dependence.

R adds one wrinkle worth stating early: `data.table`'s `:=` modifies its argument **by reference**, so a function that uses it is not pure — it changes the caller's object. The package below therefore copies before modifying, which costs a few milliseconds and buys the guarantee that calling a function twice gives the same answer twice.

Applied to the curation protocol, the decomposition is almost forced by the data flow: a function that builds the structure-key table, a function that applies one group of filters and reports how many rows each removed, a function that aggregates measurements into compounds, and a thin orchestrator that runs them in order and returns the table plus the log. Reading and writing live in a separate file that the computation never calls.



In [ ]:
aggregate_measurements <- function(frame) {
  stopifnot(all(c("inchikey", "pActivity", "censored") %in% names(frame)))
  dt <- as.data.table(frame)
  out <- dt[, {ex  <- .SD[censored == FALSE]
               use <- if (nrow(ex)) ex else .SD
               .(pActivity = as.numeric(median(use$pActivity)),
                 censored  = nrow(ex) == 0L, n_measurements = .N, n_exact = nrow(ex))},
            by = inchikey]
  setorder(out, inchikey)[]
}
demo <- data.frame(inchikey = c("A", "A", "A", "B", "B"),
                   pActivity = c(7.0, 7.4, 6.2, 5.0, 5.4),
                   censored  = c(FALSE, FALSE, TRUE, TRUE, TRUE))
print(aggregate_measurements(demo))



The printed aggregation is the unit's **worked example**, and it is small enough to verify by hand before any test is written. Compound A has three measurements, two of them exact (7.0 and 7.4) and one censored (6.2, a "greater than" record). The protocol says: when exact values exist, ignore the censored ones and take the median of the exact values.

| compound | measurements | exact values | censored values | median used | reported pActivity | `censored` flag |
|---|---|---|---|---|---|---|
| A | 3 | 7.0, 7.4 | 6.2 | median(7.0, 7.4) | 7.20 | FALSE |
| B | 2 | none | 5.0, 5.4 | median(5.0, 5.4) | 5.20 | TRUE |

Both hand-computed values appear in the output above, and both become assertions in `test-curate.R` in Section 6. Note the subtlety that the hand calculation exposes: for compound A the median of *all* three measurements would be 7.0, not 7.20 — a difference of two tenths of a log unit produced by a single word in the specification. That is exactly the kind of error the regression test of Section 8 is built to catch.

Before going further, the refactored function must be shown to reproduce the reference table, because a refactoring that changes the results is not a refactoring.



In [ ]:
refactored <- aggregate_measurements(d[, .(inchikey, pActivity, censored)])
reference  <- ref[, .(inchikey, pActivity, censored, n_measurements)]
merged <- merge(refactored, reference, by = "inchikey", suffixes = c("_new", "_ref"))
merged[, delta := pActivity_new - pActivity_ref]
g1 <- ggplot(merged, aes(pActivity_ref, pActivity_new)) +
  geom_point(size = 0.4, alpha = 0.3, colour = OKABE[1]) +
  geom_abline(slope = 1, intercept = 0, linewidth = 0.3) +
  labs(x = "reference pActivity", y = "refactored pActivity", title = "Refactored vs reference")
g2 <- ggplot(merged, aes(delta)) + geom_histogram(bins = 40, fill = OKABE[3]) +
  labs(x = "refactored - reference (log units)", y = "compounds", title = "Difference distribution")
print(g1 | g2)
cat(sprintf("compounds: refactored %d, reference %d, matched %d\n",
            nrow(refactored), nrow(reference), nrow(merged)))
cat(sprintf("max |difference| = %.2e log units (the reference file stores three decimals, so this is rounding only)\n",
            max(abs(merged$delta))))
cat(sprintf("censored flags agreeing: %.1f%%; measurement counts agreeing: %.1f%%\n",
            100 * mean(merged$censored_new == merged$censored_ref),
            100 * mean(merged$n_measurements_new == merged$n_measurements_ref)))



<p class="caption"><b>Figure 1.</b> The refactored protocol against the reference table shipped with the course. Every one of the 4,280 compounds matches to within the rounding of the stored file, and all censoring flags and measurement counts agree. The right-hand panel is the quantity that matters for a refactoring: the difference distribution is a spike at zero with a width of the order of the file's own precision, not of the data. This comparison, turned into an assertion, becomes the golden-file test of Section 6.</p>

## 3. The package on disk

An R package is a directory with a `DESCRIPTION` file, a `NAMESPACE`, source files under `R/`, documentation under `man/` and, conventionally, tests under `tests/`. Two of those are never written by hand in modern practice: `roxygen2` generates both `NAMESPACE` and `man/*.Rd` from comments beginning with `#'` that live next to the code they document<sup class="c"><a href="#r34">[34]</a></sup>. Keeping the documentation adjacent to the function is the single habit that most reliably keeps it true.

The package below is written into a temporary directory by this document. Nothing is installed into your personal library: Section 4 installs it into a temporary library that the last chunk deletes.



In [ ]:
PKG <- file.path(TMP, "rockcurate"); unlink(PKG, recursive = TRUE)
dir.create(file.path(PKG, "R"), recursive = TRUE, showWarnings = FALSE)
FILES <- list()
FILES[["DESCRIPTION"]] <- r"---(
Package: rockcurate
Title: Curation of ChEMBL Bioactivity Tables into Modelling-Ready Compound Tables
Version: 1.0.0
Authors@R: person("Laboratorio de Bioinformatica y Quimica Computacional", role = c("aut", "cre"),
    email = "lbqc@example.org")
Description: Applies the laboratory's curation protocol to a raw ChEMBL activity export:
    row filters with a per-step exclusion log, unit harmonisation to nanomolar, attachment of
    standardised structure keys, the pActivity transform and aggregation to one row per compound.
License: MIT + file LICENSE
Encoding: UTF-8
Depends: R (>= 4.1)
Imports: data.table, stats, utils, yaml
Suggests: testthat (>= 3.0.0), arrow
URL: https://github.com/<organisation>/rockcurate
BugReports: https://github.com/<organisation>/rockcurate/issues
)---"
FILES[["LICENSE"]] <- "YEAR: 2026\nCOPYRIGHT HOLDER: Laboratorio de Bioinformatica y Quimica Computacional\n"
FILES[["R/rockcurate-package.R"]] <- r"---(
#' rockcurate: curation of ChEMBL bioactivity tables
#'
#' The package applies the laboratory's curation protocol to a raw ChEMBL activity export.
#' See [curate()] for the entry point and `vignette` topics in the README.
#'
#' Importing at least one symbol from data.table is not decoration: data.table only applies
#' its own semantics inside a package that imports it ("calling environment data.table aware"),
#' and a package that merely writes `data.table::` everywhere gets data.frame behaviour,
#' silently, in expressions such as `dt[, ..cols]`.
#'
#' @keywords internal
#' @importFrom data.table := .SD .N as.data.table data.table fread fwrite setorderv setnames
#'   setcolorder setkeyv copy
#' @importFrom stats median
"_PACKAGE"
)---"
FILES[["R/config.R"]] <- r"---(
#' Parameters of the curation protocol
#'
#' Every number the protocol depends on lives here, with the defaults that reproduce the
#' laboratory's published reference table. Arguments are checked, because a silently
#' accepted nonsense value is a protocol change nobody notices.
#'
#' @param organism Character scalar; required value of the `target_organism` column.
#' @param min_heavy_atoms,max_heavy_atoms Inclusive bounds of the small-organic window.
#' @param min_value_nm,max_value_nm Inclusive bounds of a plausible concentration, in nanomolar.
#' @param drop_validity_flagged Logical; drop rows carrying a ChEMBL `data_validity_comment`.
#' @return An object of class `curation_config`.
#' @examples
#' curation_config()
#' curation_config(max_heavy_atoms = 40)
#' @export
curation_config <- function(organism = "Homo sapiens",
                            min_heavy_atoms = 3L, max_heavy_atoms = 70L,
                            min_value_nm = 1e-12, max_value_nm = 1e8,
                            drop_validity_flagged = TRUE) {
  if (!is.character(organism) || length(organism) != 1L || is.na(organism)) {
    stop("`organism` must be a single non-missing string", call. = FALSE)
  }
  if (!is.numeric(min_heavy_atoms) || !is.numeric(max_heavy_atoms) ||
      min_heavy_atoms < 1 || max_heavy_atoms < min_heavy_atoms) {
    stop("heavy-atom bounds must satisfy 1 <= min <= max", call. = FALSE)
  }
  if (!is.numeric(min_value_nm) || !is.numeric(max_value_nm) || min_value_nm <= 0 ||
      max_value_nm <= min_value_nm) {
    stop("concentration bounds must satisfy 0 < min < max", call. = FALSE)
  }
  if (!isTRUE(drop_validity_flagged) && !isFALSE(drop_validity_flagged)) {
    stop("`drop_validity_flagged` must be TRUE or FALSE", call. = FALSE)
  }
  structure(list(organism = organism,
                 min_heavy_atoms = as.integer(min_heavy_atoms),
                 max_heavy_atoms = as.integer(max_heavy_atoms),
                 min_value_nm = as.numeric(min_value_nm),
                 max_value_nm = as.numeric(max_value_nm),
                 drop_validity_flagged = drop_validity_flagged),
            class = "curation_config")
}

#' Print a curation configuration
#'
#' @param x A `curation_config`.
#' @param ... Ignored; present for compatibility with the generic.
#' @return `x`, invisibly.
#' @export
print.curation_config <- function(x, ...) {
  cat("<curation_config>\n")
  for (nm in names(x)) cat(sprintf("  %-22s %s\n", nm, format(x[[nm]])))
  invisible(x)
}

#' Read a configuration from a YAML file
#'
#' Unknown keys are an error rather than a warning: a typo in a configuration file would
#' otherwise silently leave the default in place.
#'
#' @param path Path to a YAML file with a `curation:` block, or `NULL` for the defaults.
#' @return A `curation_config`.
#' @export
read_config <- function(path = NULL) {
  if (is.null(path)) return(curation_config())
  values <- yaml::read_yaml(path)[["curation"]]
  if (is.null(values)) return(curation_config())
  known <- names(formals(curation_config))
  unknown <- setdiff(names(values), known)
  if (length(unknown)) {
    stop("unknown configuration keys: ", paste(sort(unknown), collapse = ", "), call. = FALSE)
  }
  do.call(curation_config, values)
}

#' Conversion factors from reported concentration units to nanomolar
#'
#' @format A named numeric vector.
#' @export
UNITS_TO_NM <- c(nM = 1, uM = 1e3, mM = 1e6, M = 1e9, pM = 1e-3, fM = 1e-6,
                 "nmol/L" = 1, "umol/L" = 1e3)
)---"
FILES[["R/structures.R"]] <- r"---(
#' Build a structure-key table
#'
#' R has no RDKit binding, so standardised structures are injected rather than recomputed.
#' This function expands the curated reference table into one row per ChEMBL molecule
#' identifier, which is the form the protocol joins against.
#'
#' @param curated Data frame with columns `inchikey`, `std_smiles`, `HeavyAtoms`, `chembl_ids`
#'   (the last being `|`-separated ChEMBL molecule identifiers).
#' @return A `data.table` with columns `chembl_id`, `inchikey`, `std_smiles`, `heavy_atoms`.
#' @examples
#' keys <- structure_keys(data.frame(inchikey = "AAA", std_smiles = "CCO",
#'                                   HeavyAtoms = 3, chembl_ids = "CHEMBL1|CHEMBL2"))
#' nrow(keys)
#' @export
structure_keys <- function(curated) {
  needed <- c("inchikey", "std_smiles", "HeavyAtoms", "chembl_ids")
  missing <- setdiff(needed, names(curated))
  if (length(missing)) {
    stop("the curated table is missing columns: ", paste(missing, collapse = ", "), call. = FALSE)
  }
  dt <- data.table::as.data.table(curated)[, ..needed]
  keys <- dt[, list(chembl_id = unlist(strsplit(chembl_ids, "|", fixed = TRUE))),
             by = c("inchikey", "std_smiles", "HeavyAtoms")]
  data.table::setnames(keys, "HeavyAtoms", "heavy_atoms")
  data.table::setcolorder(keys, c("chembl_id", "inchikey", "std_smiles", "heavy_atoms"))
  data.table::setkeyv(keys, "chembl_id")
  keys[]
}

#' Attach standardised structures to a measurement table
#'
#' @param frame Measurement table with a `molecule_chembl_id` column.
#' @param keys Structure-key table from [structure_keys()].
#' @param cfg A [curation_config()].
#' @param log A [curation_log()] to record the two exclusions in.
#' @return The input rows whose structure is known and inside the heavy-atom window.
#' @export
add_structures <- function(frame, keys, cfg = curation_config(), log = curation_log()) {
  dt <- data.table::as.data.table(frame)
  out <- merge(dt, keys, by.x = "molecule_chembl_id", by.y = "chembl_id",
               all.x = TRUE, sort = FALSE)
  before <- nrow(out)
  out <- out[!is.na(out$inchikey), ]
  log_step(log, "E07", "structure could not be standardised", before, nrow(out))
  before <- nrow(out)
  inside <- out$heavy_atoms >= cfg$min_heavy_atoms & out$heavy_atoms <= cfg$max_heavy_atoms
  out <- out[inside, ]
  log_step(log, "E08", "outside the small-organic window", before, nrow(out))
  out[]
}
)---"
for (nm in names(FILES)) writeLines(sub("^\n", "", FILES[[nm]]), file.path(PKG, nm))
cat(sprintf("%d files staged; package root %s\n", length(FILES), basename(PKG)))
cat(sprintf("R/config.R is %d lines, R/structures.R is %d lines\n",
            length(readLines(file.path(PKG, "R", "config.R"))),
            length(readLines(file.path(PKG, "R", "structures.R")))))



Four design decisions in those two files are worth stating explicitly, because they are the ones students most often skip.

**Configuration is a constructor with checked arguments, not a list of loose variables.** The heavy-atom window and the concentration bounds are now named arguments with a single definition, which is also the definition a Methods section can quote. `read_config()` rejects unknown keys rather than ignoring them.

**The object has a class and a `print` method.** `curation_config` is an S3 class, so `print(cfg)` shows the protocol in one readable block and a test can assert on its fields. S3 is the lightest object system in R and is the right one for a configuration record.

**Documentation sits next to the code.** The `#'` blocks become `man/*.Rd` files and the `@export` tags become the `NAMESPACE`; nothing is written twice, so nothing can disagree.

**At least one symbol is imported from `data.table`.** The package-level file carries `@importFrom data.table .SD :=` and a few more, and that line is load-bearing rather than cosmetic: `data.table` applies its own semantics only inside a package whose `NAMESPACE` imports something from it (the "calling environment data.table aware" test). A package that merely writes `data.table::` everywhere gets *data.frame* behaviour from `[`, silently — `dt[, ..cols]` then fails with "object '..cols' not found", which is exactly how this was discovered while writing Section 4.

**Dependencies are declared, and called with `::`.** `Imports: data.table, stats, utils, yaml` in `DESCRIPTION` plus `data.table::as.data.table()` in the code means the package never depends on what the user happens to have attached — the commonest reason a script that works for its author fails for everyone else.



In [ ]:
MORE <- list()
MORE[["R/curate.R"]] <- r"---(
#' A curation log
#'
#' An environment, not a list: the log is the one object in the package with reference
#' semantics, so that each step can record itself without the caller reassigning anything.
#'
#' @return An empty object of class `curation_log`.
#' @examples
#' log <- curation_log()
#' log_step(log, "E01", "example step", 10, 8)
#' as.data.frame(log)
#' @export
curation_log <- function() {
  env <- new.env(parent = emptyenv())
  env$steps <- list()
  structure(env, class = "curation_log")
}

#' Record one curation step
#'
#' @param log A [curation_log()].
#' @param code Short reason code, for example `"E03"`.
#' @param description Human-readable description of the step.
#' @param n_before,n_after Row counts before and after the step.
#' @return The log, invisibly.
#' @export
log_step <- function(log, code, description, n_before, n_after) {
  stopifnot(inherits(log, "curation_log"))
  log$steps[[length(log$steps) + 1L]] <- list(code = code, description = description,
                                              n_removed = n_before - n_after, n_rows = n_after)
  message(sprintf("%s: removed %d rows, %d remain", code, n_before - n_after, n_after))
  invisible(log)
}

#' Convert a curation log to a data frame
#'
#' @param x A [curation_log()].
#' @param ... Ignored; present for compatibility with the generic.
#' @return A data frame with one row per recorded step.
#' @export
as.data.frame.curation_log <- function(x, ...) {
  if (!length(x$steps)) return(data.frame())
  do.call(rbind, lapply(x$steps, as.data.frame))
}

#' Apply the row filters of the protocol
#'
#' @param frame Raw activity records.
#' @param cfg A [curation_config()].
#' @param log A [curation_log()].
#' @return The surviving rows.
#' @export
apply_filters <- function(frame, cfg = curation_config(), log = curation_log()) {
  dt <- data.table::as.data.table(frame)
  steps <- list(
    list("E01", "no deposited structure", function(f) !is.na(f$canonical_smiles)),
    list("E02", "target organism is not the configured one",
         function(f) !is.na(f$target_organism) & f$target_organism == cfg$organism),
    list("E03", "ChEMBL data_validity_comment flag",
         function(f) if (cfg$drop_validity_flagged) is.na(f$data_validity_comment) else rep(TRUE, nrow(f))),
    list("E04", "no numeric standard_value", function(f) !is.na(f$standard_value)))
  for (s in steps) {
    before <- nrow(dt)
    dt <- dt[s[[3]](dt), ]
    log_step(log, s[[1]], s[[2]], before, nrow(dt))
  }
  dt[]
}

#' Harmonise concentration units to nanomolar
#'
#' @param frame Rows carrying `standard_value` and `standard_units`.
#' @param cfg A [curation_config()].
#' @param log A [curation_log()].
#' @return The rows whose unit is convertible and whose value is plausible, with `value_nM`.
#' @export
harmonise_units <- function(frame, cfg = curation_config(), log = curation_log()) {
  dt <- data.table::copy(data.table::as.data.table(frame))
  dt$value_nM <- unname(UNITS_TO_NM[dt$standard_units]) * dt$standard_value
  before <- nrow(dt)
  dt <- dt[!is.na(dt$value_nM), ]
  log_step(log, "E05", "unit not convertible to nM", before, nrow(dt))
  before <- nrow(dt)
  dt <- dt[dt$value_nM >= cfg$min_value_nm & dt$value_nM <= cfg$max_value_nm, ]
  log_step(log, "E06", "implausible concentration", before, nrow(dt))
  dt[]
}

#' Derive pActivity and the censoring flag
#'
#' `pActivity` is minus the base-10 logarithm of the molar concentration; a measurement whose
#' relation is anything other than `=` is censored.
#'
#' @param frame Rows carrying `value_nM` and `standard_relation`.
#' @return The same rows with `pActivity` and `censored` added.
#' @examples
#' derive_activity(data.frame(value_nM = 1000, standard_relation = "="))$pActivity
#' @export
derive_activity <- function(frame) {
  dt <- data.table::copy(data.table::as.data.table(frame))
  dt$pActivity <- 9 - log10(dt$value_nM)
  rel <- dt$standard_relation
  dt$censored <- !(is.na(rel) | rel == "=")
  dt[]
}

#' Aggregate measurements to one row per compound
#'
#' The median of the exact measurements is used when a compound has any; compounds whose
#' measurements are all censored keep the median of the censored values and are flagged.
#' The output is sorted by InChIKey, so it is deterministic.
#'
#' @param frame Rows carrying `inchikey`, `pActivity` and `censored`.
#' @return One row per compound, with `n_measurements` and `n_exact`.
#' @examples
#' aggregate_measurements(data.frame(inchikey = c("A", "A"), pActivity = c(7, 7.4),
#'                                   censored = c(FALSE, FALSE)))
#' @export
aggregate_measurements <- function(frame) {
  needed <- c("inchikey", "pActivity", "censored")
  missing <- setdiff(needed, names(frame))
  if (length(missing)) stop("missing columns: ", paste(missing, collapse = ", "), call. = FALSE)
  dt <- data.table::as.data.table(frame)
  out <- dt[, {ex  <- .SD[.SD$censored == FALSE]
               use <- if (nrow(ex)) ex else .SD
               list(pActivity = as.numeric(stats::median(use$pActivity)),
                    censored = nrow(ex) == 0L, n_measurements = .N, n_exact = nrow(ex))},
            by = "inchikey"]
  data.table::setorderv(out, "inchikey")
  out[]
}

#' Run the whole curation protocol
#'
#' @param frame Raw activity records as delivered by the ChEMBL API.
#' @param keys Structure-key table from [structure_keys()].
#' @param cfg A [curation_config()]; the defaults reproduce the reference table.
#' @return A list with the curated `table` and the `log`.
#' @export
curate <- function(frame, keys, cfg = curation_config()) {
  log <- curation_log()
  log_step(log, "E00", "raw records as delivered", nrow(frame), nrow(frame))
  out <- apply_filters(frame, cfg, log)
  out <- harmonise_units(out, cfg, log)
  out <- add_structures(out, keys, cfg, log)
  out <- derive_activity(out)
  table <- aggregate_measurements(out)
  log_step(log, "A09", "aggregated to one row per compound", nrow(out), nrow(table))
  list(table = table, log = log)
}
)---"
MORE[["R/io.R"]] <- r"---(
#' Required columns of a raw ChEMBL activity export
#' @export
REQUIRED_COLUMNS <- c("molecule_chembl_id", "canonical_smiles", "target_organism",
                      "data_validity_comment", "standard_value", "standard_units",
                      "standard_relation")

#' Read a raw ChEMBL activity export
#'
#' The only module that touches the file system, so that every computation function can be
#' tested with a data frame built in memory.
#'
#' @param path Path to a CSV export.
#' @return A `data.table` with the required columns present.
#' @export
read_raw <- function(path) {
  if (!file.exists(path)) stop("input file does not exist: ", basename(path), call. = FALSE)
  frame <- data.table::fread(path, na.strings = c("", "NA"), showProgress = FALSE)
  missing <- setdiff(REQUIRED_COLUMNS, names(frame))
  if (length(missing)) {
    stop("input is missing required columns: ", paste(missing, collapse = ", "), call. = FALSE)
  }
  frame[]
}

#' Write a table as CSV or Parquet, chosen by the file extension
#'
#' @param frame Table to write.
#' @param path Destination; `.parquet` requires the arrow package.
#' @return `path`, invisibly.
#' @export
write_table <- function(frame, path) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  if (grepl("\\.parquet$", path)) {
    if (!requireNamespace("arrow", quietly = TRUE)) {
      stop("writing Parquet needs the arrow package", call. = FALSE)
    }
    arrow::write_parquet(frame, path)
  } else {
    data.table::fwrite(frame, path)
  }
  invisible(path)
}
)---"
MORE[["inst/scripts/curate.R"]] <- r"---(
#!/usr/bin/env Rscript
#' Command-line entry point:
#'   Rscript curate.R --input raw.csv --keys curated_reference.csv --output curated.csv
#'
#' R has no argparse in base, so the parser is fifteen lines and is kept in a function of its
#' own precisely so that it can be tested without running the protocol.
parse_args <- function(argv) {
  spec <- c("--input", "--output", "--keys", "--config", "--log")
  opts <- list(input = NULL, output = NULL, keys = NULL, config = NULL, log = NULL, verbose = FALSE)
  i <- 1L
  while (i <= length(argv)) {
    a <- argv[[i]]
    if (a == "--verbose") { opts$verbose <- TRUE; i <- i + 1L
    } else if (a %in% spec) {
      if (i == length(argv)) stop("missing value for ", a, call. = FALSE)
      opts[[sub("^--", "", a)]] <- argv[[i + 1L]]; i <- i + 2L
    } else stop("unknown argument: ", a, call. = FALSE)
  }
  if (is.null(opts$input) || is.null(opts$output) || is.null(opts$keys)) {
    stop("--input, --output and --keys are all required", call. = FALSE)
  }
  opts
}

main <- function(argv = commandArgs(trailingOnly = TRUE)) {
  opts <- tryCatch(parse_args(argv), error = function(e) {
    cat("curate:", conditionMessage(e), "\n", file = stderr()); NULL })
  if (is.null(opts)) return(1L)
  out <- tryCatch({
    raw  <- rockcurate::read_raw(opts$input)
    keys <- rockcurate::structure_keys(data.table::fread(opts$keys))
    cfg  <- rockcurate::read_config(opts$config)
    if (opts$verbose) rockcurate::curate(raw, keys, cfg)
    else suppressMessages(rockcurate::curate(raw, keys, cfg))
  }, error = function(e) { cat("curate:", conditionMessage(e), "\n", file = stderr()); NULL })
  if (is.null(out)) return(1L)
  rockcurate::write_table(out$table, opts$output)
  if (!is.null(opts$log)) rockcurate::write_table(as.data.frame(out$log), opts$log)
  cat(sprintf("curate: %d records -> %d compounds -> %s\n",
              nrow(rockcurate::read_raw(opts$input)), nrow(out$table), basename(opts$output)))
  0L
}

if (sys.nframe() == 0L) quit(status = main())
)---"
for (nm in names(MORE)) {
  dir.create(dirname(file.path(PKG, nm)), recursive = TRUE, showWarnings = FALSE)
  writeLines(sub("^\n", "", MORE[[nm]]), file.path(PKG, nm))
}
sizes <- vapply(c(names(FILES), names(MORE)),
                function(n) length(readLines(file.path(PKG, n))), integer(1))
print(data.frame(file = names(sizes), lines = as.integer(sizes)), row.names = FALSE)
cat(sprintf("\ntotal %d lines in %d files; %d of them roxygen comment lines\n", sum(sizes), length(sizes),
            sum(vapply(list.files(file.path(PKG, "R"), full.names = TRUE),
                       function(f) sum(grepl("^#'", readLines(f))), integer(1)))))



Note the three R-specific decisions in `R/curate.R`. The log is an **environment** with a class, because R copies lists on assignment: a plain list would force every step to return the log and every caller to reassign it, which is noise. The computation functions **copy before modifying**, so that `curate()` does not silently alter the caller's data. And `as.data.frame.curation_log` is an S3 **method on a standard generic**, so the log behaves like any other R object in a pipeline — the user does not have to learn a new verb.

The command-line entry point lives in `inst/scripts/`, which is where files that must be installed alongside the code but are not R source belong. After installation it is found with `system.file("scripts", "curate.R", package = "rockcurate")`, which is the only portable way to refer to it.


In [ ]:
AUX <- list()
AUX[["README.md"]] <- r"---(
 # rockcurate

Curation of ChEMBL bioactivity exports into a modelling-ready compound table, as used by the
Laboratorio de Bioinformatica y Quimica Computacional (LBQC).

## Install

    remotes::install_github("<organisation>/rockcurate")

## Use

    library(rockcurate)
    keys <- structure_keys(data.table::fread("rock1_chembl_curated.csv"))
    out  <- curate(read_raw("rock1_chembl_raw.csv"), keys)
    out$table ; as.data.frame(out$log)

or from the shell:

    Rscript curate.R --input rock1_chembl_raw.csv --keys rock1_chembl_curated.csv \
        --output rock1_curated.csv --log curation_log.csv --verbose

## What the protocol does

1. drops records without a structure, from the wrong organism, flagged by ChEMBL, or without a value;
2. harmonises units to nanomolar and drops implausible concentrations;
3. attaches standardised structures (InChIKey, parent SMILES, heavy-atom count) and keeps the
   small-organic window;
4. derives pActivity and a censoring flag;
5. aggregates to one row per compound using the median of the exact measurements.

Every exclusion is counted in the returned log. Defaults reproduce the reference table published
with the course; override them with a YAML file (see `config.yml`).

## Testing

    testthat::test_local()

## Citation

See `CITATION.cff`. Licence: MIT (code). The curated data inherit the ChEMBL CC BY-SA 3.0 licence.
)---"
AUX[["config.yml"]] <- r"---(
curation:
  organism: Homo sapiens
  min_heavy_atoms: 3
  max_heavy_atoms: 70
  max_value_nm: 1.0e+8      # YAML 1.1 wants the sign; 1.0e8 parses as a string
  drop_validity_flagged: true
)---"
AUX[["NEWS.md"]] <- r"---(
 # rockcurate 1.0.0 (2026-10-18)

* First stable release: `curate()`, the `Rscript` entry point, YAML configuration.
* Golden-file regression test against the reference ROCK1 table (4,280 compounds).

 # rockcurate 0.3.0 (2026-09-30)

* **Breaking**: `aggregate_measurements()` now uses the median of the exact measurements only;
  previously it used the median of all measurements, which biased censored-heavy compounds.
* Fixed: the heavy-atom window was applied before the structure join, so salts were judged by
  their total size.

 # rockcurate 0.2.0 (2026-09-12)

* Added: exclusion log with reason codes.
)---"
AUX[["CITATION.cff"]] <- r"---(
cff-version: 1.2.0
message: "If you use this software, please cite it as below."
title: "rockcurate: curation of ChEMBL bioactivity tables"
version: 1.0.0
date-released: "2026-10-18"
license: MIT
type: software
authors:
  - family-names: "Bustos"
    given-names: "Daniel"
    affiliation: "Universidad Catolica del Maule"
repository-code: "https://github.com/<organisation>/rockcurate"
keywords: ["cheminformatics", "ChEMBL", "data curation", "QSAR"]
)---"
AUX[["inst/CITATION"]] <- r"---(
bibentry(
  bibtype  = "Manual",
  title    = "rockcurate: Curation of ChEMBL Bioactivity Tables",
  author   = person("Daniel", "Bustos"),
  year     = "2026",
  note     = "R package version 1.0.0",
  url      = "https://github.com/<organisation>/rockcurate"
)
)---"
for (nm in names(AUX)) {
  dir.create(dirname(file.path(PKG, nm)), recursive = TRUE, showWarnings = FALSE)
  txt <- strsplit(sub("^\n", "", AUX[[nm]]), "\n", fixed = TRUE)[[1]]
  writeLines(sub("^ # ", "# ", txt), file.path(PKG, nm))   ## the leading space in the chunk above
}                                                          ## keeps this document's own headings intact
tree <- function(root, prefix = "") {
  entries <- sort(list.files(root, all.files = FALSE, full.names = TRUE))
  entries <- entries[order(!dir.exists(entries), basename(entries))]
  out <- character(0)
  for (i in seq_along(entries)) {
    last <- i == length(entries)
    out <- c(out, paste0(prefix, if (last) "`-- " else "|-- ", basename(entries[i])))
    if (dir.exists(entries[i])) out <- c(out, tree(entries[i], paste0(prefix, if (last) "    " else "|   ")))
  }
  out
}
cat("rockcurate/\n"); cat(tree(PKG), sep = "\n")



<p class="caption"><b>Listing (Section 3).</b> The repository as written by this document: the importable code under <code>R/</code>, the command-line script under <code>inst/scripts/</code>, and at the root the files that make the repository a <em>project</em> rather than a folder — package metadata in <code>DESCRIPTION</code>, a README, a licence, a NEWS file, two citation files (<code>CITATION.cff</code> for GitHub and Zenodo, <code>inst/CITATION</code> for R's own <code>citation()</code>) and a configuration example. <code>man/</code> and <code>NAMESPACE</code> are absent because they are generated, which is the next step.</p>

## 4. Documenting it, installing it, running it

`roxygen2::roxygenise()` reads the `#'` blocks and writes `NAMESPACE` and one `.Rd` file per documented object. `R CMD INSTALL` then builds the package into a library. Both are run below through `callr`, which starts a separate R process so that the installation cannot be affected by — or affect — the state of this session.



In [ ]:
suppressMessages(roxygen2::roxygenise(PKG, load_code = "source"))
LIB <- file.path(TMP, "rocklib")
dir.create(LIB, showWarnings = FALSE)
inst <- callr::rcmd("INSTALL", c(paste0("--library=", LIB), PKG), fail_on_status = FALSE, show = FALSE)
cat("NAMESPACE written by roxygen2:\n"); cat(readLines(file.path(PKG, "NAMESPACE")), sep = "\n")
cat("\nhelp pages generated:", paste(sort(list.files(file.path(PKG, "man"))), collapse = ", "), "\n")
cat("\nR CMD INSTALL exit status:", inst$status, "\n")
cat(grep("^[*]", strsplit(inst$stderr, "\n")[[1]], value = TRUE)[c(1, 2, 3)], sep = "\n")
library(rockcurate, lib.loc = LIB)
cat("\nattached rockcurate", as.character(packageVersion("rockcurate")),
    "| exported objects:", length(getNamespaceExports("rockcurate")), "\n")



The `NAMESPACE` was not written by hand, and it contains exactly two kinds of line: `export()` for the public functions and `S3method()` for the two methods, which roxygen recognised from their names without being told. The help pages exist for every exported object, which is not a courtesy — `R CMD check` fails a package whose exported function has no documentation, and that is the mechanism by which R packages are, on average, better documented than scripts.



In [ ]:
default_cfg <- curation_config()
file_cfg    <- read_config(file.path(PKG, "config.yml"))
print(default_cfg)
cat("\nconfiguration read from YAML is identical to the defaults:",
    identical(unclass(default_cfg), unclass(file_cfg)), "\n")
writeLines(c("curation:", "  max_heavy_atom: 70"), file.path(PKG, "bad.yml"))   ## note the typo
cat("a typo in the configuration file is rejected:\n  ",
    tryCatch(read_config(file.path(PKG, "bad.yml")), error = conditionMessage), "\n")
cat("an impossible window is rejected:\n  ",
    tryCatch(curation_config(min_heavy_atoms = 80, max_heavy_atoms = 3), error = conditionMessage), "\n")



Both failures are loud, and both happen before any data is read. A configuration key that is silently ignored is the most expensive kind of bug in a curation pipeline: the protocol runs, the output looks right, and it was produced with the default the user believed they had overridden.



In [ ]:
KEYS_PATH <- REF_PATH
script  <- system.file("scripts", "curate.R", package = "rockcurate", lib.loc = LIB)
out_csv <- file.path(TMP, "out", "rock1_curated.csv")
log_csv <- file.path(TMP, "out", "curation_log.csv")
t0 <- Sys.time()
run <- callr::rscript(script, cmdargs = c("--input", RAW_PATH, "--keys", KEYS_PATH,
                                          "--output", out_csv, "--log", log_csv),
                      libpath = c(LIB, .libPaths()), fail_on_status = FALSE, show = FALSE)
cat("exit status", run$status, sprintf("in %.1f s\n", as.numeric(difftime(Sys.time(), t0, units = "secs"))))
cat(run$stdout)
print(fread(log_csv))
bad_run <- callr::rscript(script, cmdargs = c("--input", file.path(TMP, "nope.csv"),
                                              "--keys", KEYS_PATH, "--output", out_csv),
                          libpath = c(LIB, .libPaths()), fail_on_status = FALSE, show = FALSE)
cat("\nmissing input -> exit status", bad_run$status, "and the message:", trimws(bad_run$stderr), "\n")



The log is the part a referee would ask for: three records lost for want of a structure, thirteen to ChEMBL's own validity flags, twenty-six without a numeric value, two whose molecule has no standardised structure, and the aggregation of 5,045 measurements into 4,280 compounds. It is produced by the library, not by this document, so the same table accompanies every run of the protocol, including the ones nobody is watching.

The last line is as important as the rest. A command-line tool that raises a traceback at a missing file is a tool that cannot be used in a pipeline; one that prints a message and exits with status 1 can be wired into a workflow manager, which is what Unit 25 does with it.

<div class="exercise"><h4>Exercise 1</h4>
<p>Add a <code>--min-heavy-atoms</code> option to the entry point that overrides the configuration file, run the protocol with the window set to 15–40 heavy atoms, and compare the exclusion log with the one above. Then decide where that option belongs: in the command line, in the YAML file, or in both — and write one sentence justifying the choice in terms of who needs to change it and how often.</p></div>

## 5. The structure of the package, drawn

Before writing tests it is worth looking at the dependency graph, because it is the thing the tests will exploit. The arrows below point from a file to the files whose functions it calls, extracted from the sources themselves. Two properties matter: the graph is acyclic, and `io.R` — the only file that touches the file system — is called by nothing inside the package. That is what "I/O at the edges" means in practice, and it is why `curate()` can be tested with a data frame built in memory.



In [ ]:
rfiles <- list.files(file.path(PKG, "R"), full.names = TRUE)
defs <- lapply(rfiles, function(f) {
  txt <- readLines(f); list(file = basename(f), txt = txt,
    names = sub("^([A-Za-z._][A-Za-z0-9._]*)\\s*<-.*$", "\\1",
                grep("^[A-Za-z._][A-Za-z0-9._]*\\s*<-", txt, value = TRUE)))
})
names(defs) <- vapply(defs, function(x) x$file, character(1))
defs <- defs[vapply(defs, function(x) length(x$names) > 0L, logical(1))]   ## the doc-only file has none
edges <- do.call(rbind, lapply(defs, function(a) do.call(rbind, lapply(defs, function(b) {
  if (a$file == b$file) return(NULL)
  used <- sum(vapply(b$names, function(n)
    any(grepl(paste0("(^|[^A-Za-z0-9._])", n, "\\("), a$txt[!grepl("^#'", a$txt)])), logical(1)))
  if (used) data.frame(from = a$file, to = b$file, n = used) else NULL
}))))
pos <- data.frame(file = c("config.R", "structures.R", "curate.R", "io.R"),
                  x = c(0.18, 0.52, 0.52, 0.86), y = c(0.25, 0.80, 0.25, 0.80))
pos$lines <- vapply(pos$file, function(f) length(defs[[f]]$txt), integer(1))
pos$funs  <- vapply(pos$file, function(f) length(defs[[f]]$names), integer(1))
e <- merge(merge(edges, pos, by.x = "from", by.y = "file"), pos, by.x = "to", by.y = "file",
           suffixes = c("_f", "_t"))
print(ggplot() +
  geom_segment(data = e, aes(x = x_f, y = y_f, xend = x_t, yend = y_t),
               arrow = arrow(length = unit(0.16, "cm"), type = "closed"),
               colour = "grey45", linewidth = 0.4) +
  geom_rect(data = pos, aes(xmin = x - 0.12, xmax = x + 0.12, ymin = y - 0.08, ymax = y + 0.08),
            fill = "#eef3fa", colour = OKABE[1], linewidth = 0.5) +
  geom_text(data = pos, aes(x, y + 0.025, label = file), fontface = "bold", size = 3) +
  geom_text(data = pos, aes(x, y - 0.04, label = sprintf("%d lines, %d objects", lines, funs)), size = 2.4) +
  annotate("text", x = 0.5, y = 0.98, size = 2.9, fontface = "italic",
           label = "arrows point from a file to the file whose functions it calls; no cycles") +
  coord_cartesian(xlim = c(0.03, 0.99), ylim = c(0.12, 1)) + theme_void())
print(e[, c("from", "to", "n")], row.names = FALSE)
cat(sprintf("\n%d files, %d lines, %d top-level objects, %d exported\n", nrow(pos), sum(pos$lines),
            sum(pos$funs), length(getNamespaceExports("rockcurate"))))



<p class="caption"><b>Figure 2.</b> Dependency graph of the package, extracted from the sources by matching every top-level object name against the call sites in the other files. <code>curate.R</code> calls <code>config.R</code> (the configuration and the unit table) and <code>structures.R</code> calls both; <code>io.R</code> is called by nothing. An import cycle — two files whose functions call each other — would appear here as a pair of opposing arrows and is the commonest structural defect in student packages; it is prevented by asking, for every new call, whether the arrow points downwards.</p>

## 6. Tests: what to assert, and how

A test suite for scientific code has four layers, and most student projects write only the first.

The framework is `testthat`<sup class="c"><a href="#r30">[30]</a></sup>, whose unit of organisation is a `test_that()` block holding several `expect_*()` assertions, and whose files are collected automatically from `tests/testthat/`.

**Unit tests** check one function against a hand-computed answer. They are fast, they localise failures precisely, and they are the only tests whose expected values you can reason about completely — like the aggregation worked by hand in Section 2.

**Tabulated tests** run the same assertion over a table of cases. In chemistry this is how edge cases are documented: a compound with one measurement, one with only censored measurements, an unknown identifier, a value at the boundary of the window. Each row of the table is a case somebody once got wrong. R has no built-in parametrisation, so the idiom is a loop that creates one `test_that()` block per case, with the case in the block's name so a failure names it.

**Property tests** assert an invariant that must hold for *every* input rather than a specific output. The protocol has two: attaching structures to a table that already has them must change nothing, and aggregating a table that already has one row per compound must return it unchanged (**idempotence**). Invariants catch classes of bug that example-based tests miss, because they do not depend on your having imagined the failing input<sup class="c"><a href="#r21">[21]</a></sup>.

**Regression (golden-file) tests** compare the output of the whole pipeline with a stored, reviewed result — here the reference table of 4,280 compounds. They are the tests that make refactoring safe, and the only ones that would have caught the protocol change recorded in the 0.3.0 entry of `NEWS.md`. Scientific software is unusually dependent on them, because for most of its functions nobody knows the right answer in advance: the literature on testing computational science calls this the *oracle problem*, and a reviewed stored output is the practical oracle<sup class="c"><a href="#r28">[28]</a></sup>.

Two mechanical points. Floating-point results are compared with a tolerance (`expect_equal` uses one by default; `expect_identical` does not), and the tolerance is a scientific decision that belongs in a comment. And randomness is tested by fixing the seed and asserting reproducibility, not by asserting a particular random value.



In [ ]:
layers <- data.frame(layer = factor(c("unit + tabulated", "property (invariants)",
                                      "CLI / integration", "golden-file regression"),
                                    levels = c("golden-file regression", "CLI / integration",
                                               "property (invariants)", "unit + tabulated")),
                     n = c(14, 4, 3, 1))
g1 <- ggplot(layers, aes(n, layer, fill = layer)) + geom_col(show.legend = FALSE, width = 0.7) +
  geom_text(aes(label = n), hjust = -0.4, size = 2.8) +
  scale_fill_manual(values = rev(OKABE[1:4])) + xlim(0, 17) +
  labs(x = "assertions the suite is designed to hold", y = NULL, title = "Layers of the suite")
stages <- data.frame(stage = factor(c("while\nwriting", "pre-commit\nhook", "CI on\npush",
                                      "code\nreview", "after\npublication"),
                                    levels = c("while\nwriting", "pre-commit\nhook", "CI on\npush",
                                               "code\nreview", "after\npublication")),
                     cost = c(1, 2, 5, 10, 500))
g2 <- ggplot(stages, aes(stage, cost, fill = cost > 100)) +
  geom_col(show.legend = FALSE, width = 0.65) + scale_y_log10() +
  scale_fill_manual(values = c(`FALSE` = OKABE[3], `TRUE` = OKABE[2])) +
  labs(x = NULL, y = "relative cost of fixing (log scale)",
       title = "Where a defect is found decides what it costs") +
  theme(axis.text.x = element_text(size = 6))
print(g1 | g2)
cat("the right-hand scale is illustrative, but its shape is not: a wrong number that reaches a\n")
cat("published paper costs a correction or a retraction, which is why the cheap layers are many\n")



<p class="caption"><b>Figure 3.</b> Left: the shape of a useful suite — many fast unit and tabulated tests, fewer invariants, a few integration tests, one regression test over the whole pipeline; the counts are the design, and the realised ones are in the table of Section 6. Right: the cost of finding a defect at each stage, on a logarithmic scale; the numbers are illustrative but the ordering is not, and it is the argument for putting the static checks in a pre-commit hook and the tests in continuous integration rather than in a reviewer's head.</p>



In [ ]:
dir.create(file.path(PKG, "tests", "testthat"), recursive = TRUE, showWarnings = FALSE)
writeLines(c('library(testthat)', 'library(rockcurate)', 'test_check("rockcurate")'),
           file.path(PKG, "tests", "testthat.R"))
TESTS <- list()
TESTS[["helper-paths.R"]] <- r"---(
## Paths come from the environment, exactly as they would in a CI job: no absolute path is
## hard-coded in a test file, and the suite can be pointed at another snapshot.
RAW_PATH <- Sys.getenv("ROCKCURATE_RAW")
REF_PATH <- Sys.getenv("ROCKCURATE_REF")
SCRIPT   <- Sys.getenv("ROCKCURATE_SCRIPT")
LIBPATH  <- Sys.getenv("ROCKCURATE_LIB")

tiny_raw <- function() {
  data.frame(
    molecule_chembl_id    = c("CHEMBL1", "CHEMBL2", "CHEMBL3", "CHEMBL1", "CHEMBL9", "CHEMBL1"),
    canonical_smiles      = c("CCO", "CC(=O)[O-].[Na+]", NA, "CCO", "c1ccccc1", "CCO"),
    target_organism       = c(rep("Homo sapiens", 5), "Rattus norvegicus"),
    data_validity_comment = c(NA, NA, NA, "Outside typical range", NA, NA),
    standard_value        = c(100, 50, 10, 1, NA, 20),
    standard_units        = rep("nM", 6),
    standard_relation     = c("=", ">", "=", "=", "=", "="),
    stringsAsFactors      = FALSE)
}

tiny_keys <- function() {
  structure_keys(data.frame(
    inchikey = c("KEY-A", "KEY-B"), std_smiles = c("CCO", "CC(=O)O"),
    HeavyAtoms = c(3, 4), chembl_ids = c("CHEMBL1", "CHEMBL2"), stringsAsFactors = FALSE))
}

reference_table <- function() data.table::fread(REF_PATH)
)---"
TESTS[["test-config.R"]] <- r"---(
test_that("the defaults are the published protocol", {
  cfg <- curation_config()
  expect_equal(c(cfg$min_heavy_atoms, cfg$max_heavy_atoms), c(3L, 70L))
  expect_identical(cfg$organism, "Homo sapiens")
  expect_equal(cfg$max_value_nm, 1e8)
  expect_s3_class(cfg, "curation_config")
})

test_that("impossible parameters are rejected at construction", {
  expect_error(curation_config(min_heavy_atoms = 80, max_heavy_atoms = 3), "heavy-atom bounds")
  expect_error(curation_config(min_value_nm = 0), "concentration bounds")
  expect_error(curation_config(organism = c("a", "b")), "single non-missing string")
  expect_error(curation_config(drop_validity_flagged = "yes"), "TRUE or FALSE")
})

test_that("a typo in a configuration file fails loudly", {
  path <- tempfile(fileext = ".yml")
  writeLines(c("curation:", "  max_heavy_atom: 70"), path)
  expect_error(read_config(path), "unknown configuration keys")
  expect_equal(read_config(NULL)$max_heavy_atoms, 70L)
})
)---"
TESTS[["test-structures.R"]] <- r"---(
test_that("structure_keys expands the pipe-separated identifiers", {
  keys <- structure_keys(data.frame(inchikey = "KEY-A", std_smiles = "CCO", HeavyAtoms = 3,
                                    chembl_ids = "CHEMBL1|CHEMBL2", stringsAsFactors = FALSE))
  expect_equal(nrow(keys), 2L)
  expect_setequal(keys$chembl_id, c("CHEMBL1", "CHEMBL2"))
  expect_equal(unique(keys$heavy_atoms), 3)
})

test_that("a curated table without the required columns is refused", {
  expect_error(structure_keys(data.frame(inchikey = "A")), "missing columns")
})

## Tabulated cases: R has no parametrisation, so one test_that per row, named by the case.
cases <- list(list(id = "CHEMBL1", keep = TRUE,  why = "known identifier, inside the window"),
              list(id = "CHEMBL9", keep = FALSE, why = "identifier absent from the key table"))
for (case in cases) {
  test_that(sprintf("add_structures: %s (%s)", case$id, case$why), {
    frame <- data.frame(molecule_chembl_id = case$id, stringsAsFactors = FALSE)
    out <- suppressMessages(add_structures(frame, tiny_keys()))
    expect_equal(nrow(out) == 1L, case$keep)
  })
}

test_that("the heavy-atom window is applied from the configuration", {
  frame <- data.frame(molecule_chembl_id = c("CHEMBL1", "CHEMBL2"), stringsAsFactors = FALSE)
  wide   <- suppressMessages(add_structures(frame, tiny_keys()))
  narrow <- suppressMessages(add_structures(frame, tiny_keys(),
                                            curation_config(min_heavy_atoms = 4)))
  expect_equal(nrow(wide), 2L)
  expect_equal(nrow(narrow), 1L)
})

test_that("property: attaching structures twice changes nothing", {
  frame <- data.frame(molecule_chembl_id = c("CHEMBL1", "CHEMBL2"), stringsAsFactors = FALSE)
  once  <- suppressMessages(add_structures(frame, tiny_keys()))
  twice <- suppressMessages(add_structures(once[, "molecule_chembl_id", drop = FALSE], tiny_keys()))
  expect_equal(twice$inchikey, once$inchikey)
})
)---"
TESTS[["test-curate.R"]] <- r"---(
test_that("aggregation matches the hand-worked example", {
  frame <- data.frame(inchikey = c("A", "A", "A", "B", "B"),
                      pActivity = c(7.0, 7.4, 6.2, 5.0, 5.4),
                      censored = c(FALSE, FALSE, TRUE, TRUE, TRUE))
  out <- as.data.frame(aggregate_measurements(frame))
  rownames(out) <- out$inchikey
  expect_equal(out["A", "pActivity"], 7.20, tolerance = 1e-9)   ## median of the exact values only
  expect_equal(out["B", "pActivity"], 5.20, tolerance = 1e-9)   ## all censored: median of all
  expect_false(out["A", "censored"])
  expect_true(out["B", "censored"])
  expect_equal(out["A", "n_exact"], 2L)
})

test_that("property: aggregation is idempotent", {
  frame <- data.frame(inchikey = c("A", "B"), pActivity = c(7.2, 5.2), censored = c(FALSE, TRUE))
  once  <- aggregate_measurements(frame)
  twice <- aggregate_measurements(as.data.frame(once)[, c("inchikey", "pActivity", "censored")])
  expect_equal(twice$pActivity, once$pActivity)
})

test_that("aggregation refuses a frame without the required columns", {
  expect_error(aggregate_measurements(data.frame(inchikey = "A")), "missing columns")
})

## Tabulated: the activity transform, three decades apart.
for (case in list(c(1, 9), c(1000, 6), c(1e6, 3))) {
  test_that(sprintf("pActivity of %g nM is %g", case[1], case[2]), {
    frame <- data.frame(value_nM = case[1], standard_relation = "=")
    expect_equal(derive_activity(frame)$pActivity, case[2], tolerance = 1e-12)
  })
}

test_that("a missing relation counts as an exact measurement", {
  frame <- data.frame(value_nM = c(10, 10, 10), standard_relation = c("=", NA, ">"))
  expect_equal(derive_activity(frame)$censored, c(FALSE, FALSE, TRUE))
})

test_that("curate on the six hand-written records leaves what we expect", {
  out <- suppressMessages(curate(tiny_raw(), tiny_keys()))
  codes <- as.data.frame(out$log)$code
  expect_true(all(c("E00", "E01", "E02", "E03", "E04", "E05", "E06", "E07", "E08", "A09") %in% codes))
  expect_equal(nrow(out$table), 2L)          ## ethanol and the acetate parent
  expect_equal(sum(out$table$n_measurements), 2L)
  expect_false(any(is.na(out$table$pActivity)))
})
)---"
TESTS[["test-golden.R"]] <- r"---(
TOLERANCE <- 1e-3        ## the reference file stores three decimals

test_that("the pipeline reproduces the reference table", {
  skip_if(RAW_PATH == "", "ROCKCURATE_RAW is not set")
  reference <- reference_table()
  out <- suppressMessages(curate(read_raw(RAW_PATH), structure_keys(reference)))
  expect_equal(nrow(out$table), nrow(reference))
  merged <- merge(as.data.frame(out$table),
                  as.data.frame(reference)[, c("inchikey", "pActivity", "censored")],
                  by = "inchikey", suffixes = c("_new", "_ref"))
  expect_equal(nrow(merged), nrow(reference))                     ## the same compounds
  expect_lt(max(abs(merged$pActivity_new - merged$pActivity_ref)), TOLERANCE)
  expect_true(all(merged$censored_new == merged$censored_ref))
})
)---"
TESTS[["test-cli.R"]] <- r"---(
test_that("the entry point writes a table and returns status 0", {
  skip_if(SCRIPT == "" || RAW_PATH == "", "the script or the data is not available")
  out <- tempfile(fileext = ".csv")
  run <- callr::rscript(SCRIPT, cmdargs = c("--input", RAW_PATH, "--keys", REF_PATH,
                                            "--output", out),
                        libpath = c(LIBPATH, .libPaths()), fail_on_status = FALSE, show = FALSE)
  expect_equal(run$status, 0L)
  expect_true(file.exists(out))
  expect_gt(nrow(data.table::fread(out)), 4000)
})

test_that("a missing input is an error message and status 1, not a traceback", {
  skip_if(SCRIPT == "", "the script is not available")
  run <- callr::rscript(SCRIPT, cmdargs = c("--input", tempfile(), "--keys", REF_PATH,
                                            "--output", tempfile(fileext = ".csv")),
                        libpath = c(LIBPATH, .libPaths()), fail_on_status = FALSE, show = FALSE)
  expect_equal(run$status, 1L)
  expect_match(run$stderr, "curate:")
})

test_that("an unknown argument is refused", {
  skip_if(SCRIPT == "", "the script is not available")
  run <- callr::rscript(SCRIPT, cmdargs = c("--input", "a.csv", "--nonsense", "1"),
                        libpath = c(LIBPATH, .libPaths()), fail_on_status = FALSE, show = FALSE)
  expect_equal(run$status, 1L)
})
)---"
for (nm in names(TESTS)) writeLines(sub("^\n", "", TESTS[[nm]]), file.path(PKG, "tests", "testthat", nm))
n_blocks <- sum(vapply(names(TESTS), function(n)
  sum(grepl("test_that\\(", readLines(file.path(PKG, "tests", "testthat", n)))), integer(1)))
cat(sprintf("wrote %d test files, %d test_that blocks (the loops add one block per case at run time)\n",
            length(TESTS), n_blocks))
cat(sprintf("assertions written: %d\n", sum(vapply(names(TESTS), function(n)
  sum(grepl("expect_", readLines(file.path(PKG, "tests", "testthat", n)))), integer(1)))))


In [ ]:
Sys.setenv(ROCKCURATE_RAW = RAW_PATH, ROCKCURATE_REF = REF_PATH,
           ROCKCURATE_SCRIPT = script, ROCKCURATE_LIB = LIB)
t0 <- Sys.time()
res <- testthat::test_dir(file.path(PKG, "tests", "testthat"), reporter = "silent",
                          load_package = "none", stop_on_failure = FALSE)
rd <- as.data.frame(res)
by_file <- aggregate(cbind(passed, failed, real) ~ file, data = rd, FUN = sum)
by_file$real <- round(by_file$real, 2)
kable(by_file, col.names = c("test file", "assertions passed", "failed", "seconds"))
cat(sprintf("\n%d test blocks, %d assertions, %d failures, %d errors, in %.1f s\n",
            nrow(rd), sum(rd$passed), sum(rd$failed), sum(rd$error),
            as.numeric(difftime(Sys.time(), t0, units = "secs"))))



Forty-odd assertions, a few seconds, and the refactoring of Section 2 is now defensible: the package reproduces the reference table to a tolerance that is stated in the test, the hand-worked aggregation is pinned, the edge cases of the structure join are documented as executable examples, and the command-line entry point is known to return the right exit statuses. Every one of those statements is re-checked automatically whenever the code changes.

The tabulated blocks are worth a second look in the table above: each case produced its own `test_that` block with the case in its name, so a failure says `pActivity of 1000 nM is 6` rather than naming a function and leaving you to find the row. That naming is why the loop idiom is preferred to several `expect_equal` calls inside one block.

## 7. Coverage: useful, and not a goal

Coverage measures which parts of the package were executed while the tests ran. It answers one question — *is there code nobody tested?* — and it answers no other. A suite can reach full coverage while asserting nothing, and a well-tested numerical routine can sit at seventy per cent because its error branches are hard to trigger. The group's rule is to look at the *uncovered* parts and decide, one by one, whether each is a missing test or an acceptable gap.

The standard R tool is `covr`, which instruments the source and reports coverage per line. It is not installed in this environment, so the chunk below measures the coarser quantity that base R can measure exactly: **function-level coverage**, obtained by `trace()`-ing every object in the package and running the suite again. It answers the same question one level up.



In [ ]:
.GlobalEnv$.rc_covered <- new.env(parent = emptyenv())   ## reachable from a traced frame
pkg_env <- as.environment("package:rockcurate")
fns <- Filter(function(n) is.function(get(n, envir = pkg_env)), ls(pkg_env))
for (f in fns) suppressMessages(trace(f, where = pkg_env, print = FALSE,
                                      tracer = bquote(assign(.(f), TRUE, envir = .GlobalEnv$.rc_covered))))
invisible(testthat::test_dir(file.path(PKG, "tests", "testthat"), reporter = "silent",
                             load_package = "none", stop_on_failure = FALSE))
for (f in fns) suppressMessages(untrace(f, where = pkg_env))
covered <- ls(.GlobalEnv$.rc_covered)
cov <- data.frame(object = fns, exercised = fns %in% covered)
cov$file <- vapply(cov$object, function(o) {
  hit <- names(Filter(function(d) o %in% d$names, defs)); if (length(hit)) hit[1] else "other"},
  character(1))
by_mod <- aggregate(exercised ~ file, data = cov, FUN = function(v) 100 * mean(v))
print(ggplot(by_mod, aes(exercised, reorder(file, exercised), fill = exercised >= 80)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%.0f%%", exercised)), hjust = -0.2, size = 2.8) +
  geom_vline(xintercept = 80, linetype = 2, linewidth = 0.4) +
  scale_fill_manual(values = c(`FALSE` = OKABE[2], `TRUE` = OKABE[3])) + xlim(0, 118) +
  labs(x = "exported objects exercised by the suite (%)", y = NULL,
       title = sprintf("Function-level coverage (total %.0f%%)", 100 * mean(cov$exercised))))
print(cov[!cov$exercised, c("object", "file")], row.names = FALSE)
cat(sprintf("\n%d of %d exported objects were called while the suite ran\n",
            sum(cov$exercised), nrow(cov)))



<p class="caption"><b>Figure 4.</b> Fraction of each file's exported objects that the suite actually called, measured by tracing the package namespace while the tests ran, with the project's 80 % threshold. The gaps are informative rather than embarrassing: the untouched objects are the ones listed below the figure, and each is either a missing test or a deliberate decision. Line-level coverage needs <code>covr</code>, which is not installed here; the question it answers is the same one, asked of lines instead of functions.</p>

## 8. The bug that passes most of the tests

Now the experiment this unit exists for. `NEWS.md` records a real protocol change at version 0.3.0: aggregation used to take the median of *all* measurements of a compound and was changed to take the median of the exact ones only. Suppose a well-meaning refactoring reverted it — a single line, `use <- if (nrow(ex)) ex else .SD` becoming `use <- .SD`. The code still runs, the output still has the right shape, and most of the test suite still passes.

The chunk below writes a copy of the package with exactly that change, installs it into a *second* temporary library and runs the same suite against it in a separate R process. The separate process is not a precaution, it is a necessity: `rockcurate` is already attached in this session, and R caches a namespace once loaded, so editing the file on disk — or even reinstalling over it — does not change the functions this session will call.



In [ ]:
BUG <- file.path(TMP, "rockcurate_bug")
unlink(BUG, recursive = TRUE)
dir.create(BUG, showWarnings = FALSE)
invisible(file.copy(list.files(PKG, full.names = TRUE, all.files = TRUE, no.. = TRUE),
                    BUG, recursive = TRUE))
target <- file.path(BUG, "R", "curate.R")
src <- readLines(target)
hit <- grep("use <- if (nrow(ex)) ex else .SD", src, fixed = TRUE)
stopifnot(length(hit) == 1L)
src[hit] <- sub("use <- if (nrow(ex)) ex else .SD", "use <- .SD        ## BUG introduced on purpose",
                src[hit], fixed = TRUE)
writeLines(src, target)
BUGLIB <- file.path(TMP, "buglib"); dir.create(BUGLIB, showWarnings = FALSE)
invisible(callr::rcmd("INSTALL", c(paste0("--library=", BUGLIB), BUG), fail_on_status = FALSE, show = FALSE))
bug_res <- callr::r(function(dir, lib, env) {
    do.call(Sys.setenv, as.list(env))
    library(rockcurate, lib.loc = lib)
    rd <- as.data.frame(testthat::test_dir(dir, reporter = "silent", load_package = "none",
                                           stop_on_failure = FALSE))
    rd[, c("file", "test", "passed", "failed", "error")]
  }, args = list(dir = file.path(BUG, "tests", "testthat"), lib = BUGLIB,
                 env = list(ROCKCURATE_RAW = RAW_PATH, ROCKCURATE_REF = REF_PATH,
                            ROCKCURATE_SCRIPT = "", ROCKCURATE_LIB = BUGLIB)),
  libpath = c(BUGLIB, .libPaths()))
failing <- bug_res[bug_res$failed > 0 | bug_res$error, c("file", "test", "failed")]
kable(failing, row.names = FALSE, col.names = c("file", "failing test block", "assertions failed"))
cat(sprintf("\n%d of %d test blocks fail with the buggy aggregation; %d still pass\n",
            nrow(failing), nrow(bug_res), nrow(bug_res) - nrow(failing)))
cat("files that still pass entirely:",
    paste(setdiff(unique(bug_res$file), unique(failing$file)), collapse = ", "), "\n")


In [ ]:
tidy_pkg <- d[, .(inchikey, pActivity, censored)]
good <- aggregate_measurements(tidy_pkg)
bad  <- tidy_pkg[, .(pActivity = as.numeric(median(pActivity))), by = inchikey]
cmp  <- merge(good[, .(inchikey, good = pActivity)], bad[, .(inchikey, bad = pActivity)], by = "inchikey")
cmp[, delta := bad - good]
affected <- cmp[abs(delta) > 1e-9]
g1 <- ggplot(affected, aes(delta)) + geom_histogram(bins = 30, fill = OKABE[2]) +
  labs(x = "buggy - correct pActivity (log units)", y = "compounds",
       title = sprintf("%d of %d compounds changed", nrow(affected), nrow(cmp)))
layer_df <- data.frame(
  layer = factor(c("test-structures.R", "test-config.R", "test-cli.R", "test-curate.R", "test-golden.R"),
                 levels = c("test-structures.R", "test-config.R", "test-cli.R",
                            "test-curate.R", "test-golden.R")),
  caught = as.integer(c("test-structures.R", "test-config.R", "test-cli.R",
                        "test-curate.R", "test-golden.R") %in% failing$file))
g2 <- ggplot(layer_df, aes(layer, caught, fill = caught == 1)) +
  geom_col(width = 0.6, show.legend = FALSE) +
  scale_fill_manual(values = c(`FALSE` = OKABE[3], `TRUE` = OKABE[2])) +
  scale_y_continuous(breaks = c(0, 1), labels = c("passed", "FAILED"), limits = c(0, 1.2)) +
  labs(x = NULL, y = NULL, title = "Which test file caught the bug") +
  theme(axis.text.x = element_text(size = 5.6, angle = 20, hjust = 1))
print(g1 | g2)
cat(sprintf("compounds whose potency changed: %d (%.1f%%)\n", nrow(affected),
            100 * nrow(affected) / nrow(cmp)))
cat(sprintf("median absolute change among them: %.3f log units; largest: %.3f\n",
            median(abs(affected$delta)), max(abs(affected$delta))))
cat(sprintf("compounds moved across the pActivity = 7 potency boundary: %d\n",
            sum((cmp$good >= 7) != (cmp$bad >= 7))))



<p class="caption"><b>Figure 5.</b> The cost of a one-line regression. Left: the distribution of the change in reported potency over the compounds it affects — about one per cent of the data set, with a median absolute change of half a log unit and a tail several times the experimental error of the assay. Right: which test file noticed. The configuration tests, the structure tests and the command-line tests all pass, because none of them exercises the rule that changed; the hand-worked aggregation test and the golden-file regression test fail.</p>

Three lessons follow, and they are the reason the group insists on regression tests for pipelines.

First, **a bug that changes a scientific result need not break anything**. The buggy pipeline produces a table of exactly the right size with plausible values, and touches only the small fraction of compounds that have both exact and censored measurements — which is precisely why nobody would notice. Without a stored reference, the only way to catch it would be for somebody to re-derive a number by hand.

Second, **test layers are not interchangeable**. The property tests are valuable and they could not have caught this, because idempotence holds for the buggy rule too. A suite with only property tests, or only unit tests of helper functions, has a hole exactly where the science lives.

Third, **the golden file must be reviewed, not merely stored**. A regression test pins the output to whatever was there when it was written; if the stored table had itself been produced by the buggy code, the test would now enforce the bug. The reference table of this course was checked against a published protocol and a hand-worked example before being frozen, and that check is what gives the test its authority.

<div class="exercise"><h4>Exercise 2</h4>
<p>Introduce a second bug of your own — change the heavy-atom comparison from <code>&gt;=</code> to <code>&gt;</code>, or swap <code>9 - log10(value_nM)</code> for <code>9 - log(value_nM)</code> — and predict, before running, which test blocks will fail. Then run the suite. The interesting case is the bug that fails <em>no</em> test: write the test that would have caught it, and add it to <code>tests/testthat/</code>.</p></div>

## 9. Static checks, formatting and pre-commit

Style is not an aesthetic question in a shared repository; it is a question of how much of a review is spent on things a machine could have said. The R ecosystem's tools are `lintr` (static checks) and `styler` (automatic formatting), configured by a `.lintr` file at the repository root, plus `R CMD check` itself, which is a far stricter static analyser than anything available in most languages: it checks documentation against code, finds undeclared dependencies, runs the examples and the tests, and refuses a package whose exported function is undocumented.

Neither `lintr` nor `styler` is installed in this environment, so the chunk below writes the configuration the group uses and then runs a **twenty-line static checker written here with base R's own parser**, which finds the four defects that cost the group the most review time. `getParseData()` gives the full token stream of a parsed file, which is the same mechanism `lintr` is built on.



In [ ]:
writeLines(c("linters: linters_with_defaults(", "    line_length_linter(100),",
             "    object_name_linter(styles = c('snake_case', 'symbols')),",
             "    T_and_F_symbol_linter(), undesirable_function_linter()", "  )",
             "exclusions: list('tests')"), file.path(PKG, ".lintr"))
check_style <- function(path) {
  lines <- readLines(path)
  pd <- utils::getParseData(parse(path, keep.source = TRUE))
  hit <- function(where, rule) if (length(where)) data.frame(line = where, rule = rule) else NULL
  findings <- do.call(rbind, list(
    hit(which(nchar(lines) > 100), "line longer than 100 characters"),
    hit(pd$line1[pd$token == "SYMBOL" & pd$text %in% c("T", "F")], "T / F instead of TRUE / FALSE"),
    hit(pd$line1[pd$token == "EQ_ASSIGN"], "= used for assignment"),
    hit(grep("^\\s*(library|require)\\(", lines), "library() inside a package file")))
  if (is.null(findings)) data.frame(line = integer(0), rule = character(0)) else
    findings[order(findings$line), ]
}
clean <- do.call(rbind, lapply(list.files(file.path(PKG, "R"), full.names = TRUE), function(f) {
  found <- check_style(f)
  if (nrow(found)) data.frame(file = basename(f), found) else NULL
}))
cat("files checked:", length(list.files(file.path(PKG, "R"))),
    "| findings in the package source:", if (is.null(clean)) 0 else nrow(clean), "\n")
if (!is.null(clean)) print(clean, row.names = FALSE)
sloppy <- file.path(TMP, "sloppy_example.R")
writeLines(c("library(data.table)", "f = function( x ){",
             "  flag = T", paste0("  y = ", paste(rep("x", 40), collapse = " + ")),
             "  if (flag) y else NULL", "}"), sloppy)
cat("\nthe same checker on a deliberately sloppy file:\n")
print(check_style(sloppy), row.names = FALSE)



The package source yields exactly one finding — a line of `curate.R` past the hundred-character limit — and that is the honest result of a first draft written with the rules in mind but not enforced: the violation is real, it is trivial, and it would be fixed before the pull request rather than argued about in it. The sloppy file is the teaching part: a `library()` call inside package code (which would attach a package into the *user's* session as a side effect of loading yours — the reason `DESCRIPTION` and `::` exist), `T` instead of `TRUE` (`T` is an ordinary variable in R and can be reassigned, which has produced genuinely mysterious bugs), `=` used for assignment, and a line nobody can read in a side-by-side diff. None of them is a scientific error; all of them cost reviewer attention, which is the scarcest resource in the group.

`pre-commit` closes the loop: a `.pre-commit-config.yaml` with the `lintr`, `styler` and `parsable-R` hooks from `lorenzwalthert/precommit`, plus `check-added-large-files` with a 500 kB limit — the last being the hook that has saved the group most often, because the commonest way to ruin a repository is to commit a 200 MB trajectory. The hooks run on `git commit`, so the main branch never contains code that fails the checks, and a reviewer never spends a comment on formatting.

## 10. Continuous integration

Continuous integration runs the test suite on a clean machine, on every push, on the R versions the project claims to support. It converts "it works on my laptop" into evidence, and extended versions of the idea re-run whole analyses on every commit so that a paper's figures cannot drift from its code<sup class="c"><a href="#r12">[12]</a></sup>. It also presupposes the practice this unit takes for granted: the project lives in version control, with branches and pull requests<sup class="c"><a href="#r11">[11]</a></sup>. The workflow below is the one the group uses for R packages; it is written here and explained, not executed, because this document has no GitHub runner.



In [ ]:
CI <- r"---(
name: R-CMD-check

on:
  push:
    branches: [main]
  pull_request:

jobs:
  check:
    runs-on: ${{ matrix.config.os }}
    name: ${{ matrix.config.os }} (R ${{ matrix.config.r }})
    strategy:
      fail-fast: false
      matrix:
        config:
          - {os: ubuntu-latest,  r: 'release'}
          - {os: ubuntu-latest,  r: 'oldrel-1'}
          - {os: macos-latest,   r: 'release'}
          - {os: windows-latest, r: 'release'}
    env:
      R_KEEP_PKG_SOURCE: yes
    steps:
      - uses: actions/checkout@v4
      - uses: r-lib/actions/setup-r@v2
        with:
          r-version: ${{ matrix.config.r }}
          use-public-rspm: true
      - uses: r-lib/actions/setup-r-dependencies@v2
        with:
          extra-packages: any::rcmdcheck, any::lintr, any::covr
          needs: check
      - uses: r-lib/actions/check-r-package@v2
        with:
          args: 'c("--no-manual", "--as-cran")'
          error-on: '"warning"'
      - name: Lint
        run: lintr::lint_package()
        shell: Rscript {0}
      - name: Coverage
        if: matrix.config.os == 'ubuntu-latest' && matrix.config.r == 'release'
        run: covr::codecov(quiet = FALSE)
        shell: Rscript {0}
)---"
dir.create(file.path(PKG, ".github", "workflows"), recursive = TRUE, showWarnings = FALSE)
writeLines(sub("^\n", "", CI), file.path(PKG, ".github", "workflows", "R-CMD-check.yaml"))
cat(sub("^\n", "", CI))
cat(sprintf("\nthe workflow is %d lines and declares %d jobs per push\n",
            length(strsplit(trimws(CI), "\n")[[1]]), 4))


In [ ]:
mat <- data.frame(os = c("ubuntu-latest", "ubuntu-latest", "macos-latest", "windows-latest"),
                  r  = c("release", "oldrel-1", "release", "release"),
                  minutes = c(4, 4, 6, 8))
mat$job <- sprintf("%s\nR %s", mat$os, mat$r)
print(ggplot(mat, aes(x = reorder(job, -minutes), y = minutes, fill = os)) +
  geom_col(width = 0.6, show.legend = FALSE) +
  geom_text(aes(label = sprintf("%d min", minutes)), vjust = -0.4, size = 2.6) +
  scale_fill_manual(values = c("ubuntu-latest" = OKABE[1], "macos-latest" = OKABE[3],
                               "windows-latest" = OKABE[4])) +
  ylim(0, 10) + labs(x = NULL, y = "typical wall-clock time",
                     title = "The R-CMD-check matrix: four jobs on every push") +
  theme(axis.text.x = element_text(size = 6)))
cat("a matrix of three operating systems and two R versions catches the three commonest\n")
cat("portability bugs: a path built with '/' or a case-insensitive filename, a package that is\n")
cat("not available on one platform, and a base-R function whose behaviour changed between versions\n")
cat("(the times are typical for a package of this size; they are not measured here)\n")



<p class="caption"><b>Figure 6.</b> The continuous-integration matrix declared in the workflow: four jobs, each installing the package and its declared dependencies from scratch, running <code>R CMD check --as-cran</code>, the linter and — on one job only — the coverage report. The bar heights are typical wall-clock times for a package of this size rather than measurements; <code>fail-fast: false</code> keeps the other three running when one fails, which is what makes the matrix diagnostic rather than merely binary.</p>

Three details in that YAML carry most of its value. `setup-r-dependencies` installs what `DESCRIPTION` declares, so a missing entry in `Imports:` fails the build rather than hiding behind the author's library. `error-on: '"warning"'` turns an `R CMD check` warning — an undocumented argument, an undeclared dependency, a non-portable file name — into a failed build instead of a line nobody reads. And running on pull requests means a reviewer sees a green tick before reading a line, which changes the conversation from "does it work" to "is it right". Journals increasingly check the same things the workflow does, and the journals whose policies require working code are measurably the ones whose papers can be rerun<sup class="c"><a href="#r14">[14]</a></sup>. Pinning the environment itself — a `renv.lock`, a conda specification or a container built from it — is what makes the clean machine reproducible rather than merely clean<sup class="c"><a href="#r32">[32]</a><a href="#r13">[13]</a></sup>.

## 11. Versions, NEWS and citation

A package that other work depends on needs a version number that means something. **Semantic versioning** encodes compatibility in three integers: `MAJOR.MINOR.PATCH`, where a patch fixes a bug without changing behaviour, a minor release adds functionality compatibly, and a major release breaks something. For research software there is a fourth rule the group adds: **a change that alters published numbers is a major release even if the interface is unchanged** — the 0.3.0 entry in this package's `NEWS.md` would have been 1.0.0 under that rule had the package been stable.



In [ ]:
news <- readLines(file.path(PKG, "NEWS.md"))
heads <- grep("^# rockcurate", news, value = TRUE)
rel <- data.frame(version = sub("^# rockcurate ([0-9.]+).*$", "\\1", heads),
                  date = sub("^.*\\(([0-9-]+)\\).*$", "\\1", heads))
grp <- cumsum(grepl("^# rockcurate", news))
bodies <- split(news, grp)[as.character(seq_len(max(grp)))]   ## one block per release heading
rel$breaking <- vapply(bodies, function(b) any(grepl("Breaking", b)), logical(1))
rel$kinds <- vapply(bodies, function(b) {
  k <- unique(sub("^\\* \\*?\\*?([A-Za-z]+).*$", "\\1", grep("^\\* ", b, value = TRUE)))
  paste(k, collapse = ", ")}, character(1))
rel$x <- rev(seq_len(nrow(rel)))
print(ggplot(rel, aes(x, 0)) +
  geom_line(aes(group = 1), colour = "grey70", linewidth = 0.4) +
  geom_point(aes(colour = breaking), size = 4, show.legend = FALSE) +
  geom_text(aes(y = 0.22, label = paste0("v", version)), fontface = "bold", size = 3) +
  geom_text(aes(y = -0.22, label = date), size = 2.5) +
  geom_text(aes(y = -0.42, label = kinds), size = 2.3, colour = "grey30") +
  geom_text(data = rel[rel$breaking, ], aes(y = 0.45, label = "changes published numbers"),
            size = 2.4, colour = "#b32d00") +
  scale_colour_manual(values = c(`FALSE` = OKABE[1], `TRUE` = OKABE[2])) +
  coord_cartesian(ylim = c(-0.6, 0.62)) + theme_void() +
  labs(title = "Release history read from NEWS.md") +
  theme(plot.title = element_text(face = "bold", size = 9)))
print(rel[, c("version", "date", "breaking", "kinds")], row.names = FALSE)
cat("\ncitation('rockcurate') reads inst/CITATION and prints:\n")
print(utils::citation("rockcurate", lib.loc = LIB))



<p class="caption"><b>Figure 7.</b> The release history, parsed from the package's own <code>NEWS.md</code>. The orange release is the one that changed scientific output — the aggregation rule — and is flagged as such, because a paper that used version 0.2.0 and a paper that used version 0.3.0 report different numbers from the same raw data. A version number in a Methods section is only useful if the NEWS file says what changed.</p>

R packages carry their citation in two places, and the package written here has both. `inst/CITATION` is read by `citation("rockcurate")`, which is the command every R user already knows and the one that produces the BibTeX entry printed above. `CITATION.cff` is the ecosystem-neutral file that GitHub renders as a "Cite this repository" button and that Zenodo reads when archiving a release<sup class="c"><a href="#r5">[5]</a><a href="#r6">[6]</a></sup>. Software citation matters for two reasons: it is the only credit a methods developer receives, and it pins the exact version a result depends on. The group's rule is that a paper cites the software it used with its version, and that the group's own software carries both citation files and an archived DOI from the first release that produced a published number.

The licence decision is equally mechanical. Code goes out under MIT unless a dependency requires otherwise; data derived from ChEMBL keeps CC BY-SA 3.0, which propagates to derived tables; and a repository with no licence is, legally, a repository nobody may reuse — the default is "all rights reserved", not "public domain". R enforces part of this: `R CMD check` warns about a `License:` field it does not recognise, and CRAN refuses the package. Taken together, a versioned repository with a licence, a citation file, a NEWS file and an archived release is what the FAIR principles for research software ask for<sup class="c"><a href="#r7">[7]</a></sup>, and it is also the minimum that makes a project open to contributions from outside the group rather than readable only<sup class="c"><a href="#r8">[8]</a></sup>.

## 12. Documentation in four layers

Documentation fails when it is treated as one thing. It is four, each answering a different question and each with a different audience and lifetime<sup class="c"><a href="#r19">[19]</a><a href="#r20">[20]</a></sup>. In R the four map onto artefacts the tooling already knows about: `README.md`, the roxygen help pages, a vignette, and a `pkgdown` reference site generated from the first three.



In [ ]:
exported <- getNamespaceExports("rockcurate")
rd_files <- sub("\\.Rd$", "", list.files(file.path(PKG, "man")))
docs <- data.frame(
  layer = c("README.md", "roxygen help pages", "vignette", "reference site"),
  question = c("Why does this exist and how do I run it in five minutes?",
               "What does this function do, what does it take and return?",
               "How do I do the thing I came for, end to end?",
               "What is the complete surface of the package?"),
  audience = c("a new user, or you in six months", "someone calling the function",
               "a student starting a project", "an advanced user; generated"),
  size = c(length(unlist(strsplit(readLines(file.path(PKG, "README.md")), "\\s+"))),
           length(rd_files), 0L, length(exported)))
kable(docs, col.names = c("layer", "question it answers", "audience", "size (words / count)"))
per_file <- data.frame(file = names(defs),
                       objects = vapply(defs, function(x) length(x$names), integer(1)))
per_file$documented <- vapply(names(defs), function(f)
  sum(defs[[f]]$names %in% rd_files | vapply(defs[[f]]$names, function(n)
    any(grepl(paste0("^", n, "$"), rd_files)), logical(1))), integer(1))
per_file$exported <- vapply(names(defs), function(f) sum(defs[[f]]$names %in% exported), integer(1))
long <- rbind(data.frame(file = per_file$file, what = "top-level objects", n = per_file$objects),
              data.frame(file = per_file$file, what = "exported", n = per_file$exported),
              data.frame(file = per_file$file, what = "with a help page", n = per_file$documented))
print(ggplot(long, aes(file, n, fill = what)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.7) +
  scale_fill_manual(values = c("top-level objects" = "grey70", "exported" = OKABE[1],
                               "with a help page" = OKABE[3])) +
  labs(x = NULL, y = "count", fill = NULL, title = "Documentation coverage by file"))
cat(sprintf("%d exported objects, %d help pages, %d vignettes (the vignette is the layer this\n",
            length(exported), length(rd_files), 0))
cat("package is missing, and the one a new student would read first)\n")



<p class="caption"><b>Figure 8.</b> Top-level objects, exported objects and generated help pages per file. Every exported object has a help page, because <code>roxygen2</code> generated one from the comment block beside it and <code>R CMD check</code> would fail otherwise; the internal objects deliberately have none. The missing layer is the vignette, which is the honest state of a package of this age: a README and complete help pages, no end-to-end narrative yet.</p>

A README that takes five minutes to read is worth more than a reference site nobody generates. The order of effort is: README first, then roxygen blocks on the exported functions with at least one runnable `@examples` line each (R runs them during `R CMD check`, so an example that breaks is a failed build — documentation that is tested), then a vignette that solves one real task end to end, and only then a `pkgdown` site if the package has enough surface to need one.

## 13. Reviewing code, and being reviewed

Code review in a laboratory is not quality control by a superior; it is two people making sure that a piece of work can be defended. Industrial studies of review practice find that its measured value lies less in defect detection than in knowledge transfer and in the readability of what survives<sup class="c"><a href="#r22">[22]</a><a href="#r23">[23]</a></sup> — which is precisely what a research group needs when a student leaves.



In [ ]:
review <- data.frame(
  area = c("Correctness", "Correctness", "Correctness", "Scientific validity", "Scientific validity",
           "Readability", "Readability", "Structure", "Structure", "Hygiene", "Reproducibility"),
  question = c("Does a test cover the behaviour that changed? Does the golden test still pass?",
               "Are floating-point comparisons made with a stated tolerance?",
               "Are edge cases handled: empty input, all values censored, a single compound?",
               "Is the change to the protocol recorded in NEWS.md and in the Methods?",
               "Do the defaults still reproduce the published reference table?",
               "Can the function be understood without reading its callers? Is the name honest?",
               "Is there a roxygen block saying what it takes, returns and errors on?",
               "Is I/O at the edges? Does the call graph stay acyclic?",
               "Are magic numbers in the configuration rather than in the expressions?",
               "Does the static check pass? Any leftover print(), commented-out code or large files?",
               "Are seeds fixed and dependencies declared where the result depends on them?"))
kable(review, col.names = c("area", "question the reviewer asks"))
cat(sprintf("\n%d items; the first five are the ones worth a conversation, the rest should have been\n",
            nrow(review)))
cat("caught by the static check or the tests before the pull request was opened\n")



The etiquette matters as much as the checklist. Review the code, not the author: "this function reads the file twice" rather than "you did not think about efficiency". Distinguish a blocking objection from a preference, and say which it is — the convention is to prefix non-blocking remarks with "nit:". Ask questions when you do not understand instead of assuming an error. Approve when the change is an improvement, not when it is perfect; a review that demands perfection stops work. And on the receiving side: a review comment is information about your code, not about you, and the correct response to a good catch is to say so and fix it.

<div class="box bridge"><h4>Bridge</h4>
<p>The protocol packaged here was developed in <a href="../U6_Data_Quality/index.html">Unit 6</a> and its raw data described in <a href="../U3_Getting_Data/index.html">Unit 3</a>; the reproducibility practices around it — version control, environments, provenance — are <a href="../U2_Reproducibility/index.html">Unit 2</a>, and the tabular idioms used throughout are <a href="../U4_Tidy_Data/index.html">Unit 4</a>. The numbers this package produces are reported following <a href="../U23_Communication/index.html">Unit 23</a>, scaled to a cluster and wrapped in a workflow manager in <a href="../U25_Scaling_HPC/index.html">Unit 25</a>, and served to collaborators as an application in <a href="../U26_Apps_Deployment/index.html">Unit 26</a>; the capstone, <a href="../U27_Capstone/index.html">U27</a>, expects the project to be delivered as a package of this shape. In the sister courses, <a href="../../../Cheminformatics/U7_Data_Curation/index.html">Data curation</a> develops the chemistry of the standardisation steps whose output is injected here, <a href="../../../Cheminformatics/U4_Standardisation/index.html">Standardisation</a> explains why the largest-fragment and uncharging operations are ordered as they are, and the bioinformatics course's <a href="../../../BGM324/01_Sequence_Alignment/index.html">Sequence alignment</a> unit provides a second kind of pipeline — command-line tools glued together — on which the same packaging discipline applies. The Python edition of this unit, which builds the same protocol as a Python package with pytest and ruff, is <a href="../../Python/U24_Software_Engineering/index.html">here</a>.</p></div>

## 14. Summary

**A notebook is a bench, a package is an instrument.** The conversion is mechanical: pure functions for the computation, I/O at the edges, parameters in a checked configuration object, a thin `Rscript` entry point, and a `DESCRIPTION` that makes the whole thing installable<sup class="c"><a href="#r1">[1]</a><a href="#r2">[2]</a></sup>.

**Refactoring is only safe with a reference.** The function extracted in Section 2 reproduced the laboratory's curated table to within the precision of the stored file, and that comparison became the regression test that guards every later change.

**Four layers of tests, each catching what the others cannot.** Unit tests against hand-computed answers, tabulated tests over edge cases, property tests for invariants such as idempotence, and golden-file regression tests for the pipeline as a whole<sup class="c"><a href="#r21">[21]</a></sup>. The injected one-line bug of Section 8 passed the configuration, structure and command-line tests and was caught only by the hand-worked aggregation test and the golden file — after changing the reported potency of about one per cent of the compounds by a median of half a log unit.

**Coverage is a diagnostic, not a target.** Read the uncovered objects; a suite can reach any percentage while asserting nothing.

**Automate the arguments away.** `lintr`, `styler` and pre-commit remove style from review; a four-job `R-CMD-check` matrix removes "works on my machine"; `error-on: warning` removes the slow erosion of the checks.

**Make the software citable and the changes legible.** Semantic versioning with a research-specific rule — anything that changes published numbers is a breaking change — a `NEWS.md` that says what changed, a licence, `inst/CITATION` and `CITATION.cff`<sup class="c"><a href="#r5">[5]</a><a href="#r6">[6]</a></sup>.

**Review is knowledge transfer.** The checklist finds defects; the conversation is what keeps a pipeline usable after its author has graduated<sup class="c"><a href="#r22">[22]</a></sup>.

## 15. Self-assessment

<details><summary>1. Why does an R package put its code in <code>R/</code> and its documentation in <code>man/</code> instead of keeping everything in one script?</summary>
<p>Because the directory layout is a contract that the tooling enforces. <code>R CMD INSTALL</code> knows where to find the code, <code>R CMD check</code> can pair every exported object with its help page and fail when one is missing, <code>roxygen2</code> can regenerate <code>man/</code> and <code>NAMESPACE</code> from the comments beside the code, and the user gets <code>?curate</code> for free. A script offers none of these, which is why a script cannot be checked.</p></details>

<details><summary>2. What makes a function "pure", and why does purity make testing cheap? What does <code>data.table</code> do to purity?</summary>
<p>Its result depends only on its arguments, and it changes nothing outside itself: no global state, no file reads, no side effects. Such a function is tested by calling it and comparing the return value — no fixtures, no temporary files, no ordering between tests. <code>data.table</code>'s <code>:=</code> modifies its argument in place, so a function using it silently changes the caller's object; the package therefore calls <code>data.table::copy()</code> before modifying, which is why <code>harmonise_units()</code> can be called twice with the same answer.</p></details>

<details><summary>3. Give the hand-worked aggregation example and the number the buggy version would produce.</summary>
<p>Compound A has measurements 7.0 and 7.4 (exact) and 6.2 (censored). The protocol takes the median of the exact values, 7.20. The reverted rule takes the median of all three, 7.0. The difference is 0.2 log units for that compound, and across the real data set the bug changed the potency of about one per cent of the compounds by a median of half a log unit, with a tail several times the experimental error, moving compounds across the potency threshold used for classification.</p></details>

<details><summary>4. A colleague says their suite has 100 % coverage, so the code is correct. Respond.</summary>
<p>Coverage measures which code ran, not whether anything was checked. A suite that calls every function and asserts nothing reaches 100 %. Conversely, the objects not exercised in Section 7 are a short, inspectable list, and for each one the question is "missing test or acceptable gap?". Coverage is useful read item by item and misleading as a single number to maximise.</p></details>

<details><summary>5. Which kind of test would have caught a change in the unit-conversion table, and which would not?</summary>
<p>The tabulated test asserting that 1 nM gives pActivity 9 and 1,000 nM gives 6 catches it immediately. The golden-file test catches it too, but only reports that the whole pipeline moved. Property tests do not: idempotence and determinism hold for the wrong conversion as well. This is why the suite has layers — the unit test localises, the golden test guarantees.</p></details>

<details><summary>6. Why did the buggy package have to be installed into a second library and tested in a separate process?</summary>
<p>Because <code>rockcurate</code> was already attached in this session, and R caches a namespace once it is loaded: editing the file, or even reinstalling the package over itself, does not change the functions the running session will call. A second library plus a child process started by <code>callr</code> gives an unambiguous answer. The same reasoning applies to any "I fixed it" claim made without restarting the session.</p></details>

<details><summary>7. What belongs in a configuration file, and what belongs in the code?</summary>
<p>Values a user may legitimately change for their project — the organism, the heavy-atom window, the plausibility bounds — belong in configuration, with defaults in the code that reproduce the published protocol. The structure of the protocol itself, the order of the filters and the aggregation rule do not: making them configurable would turn a reproducible protocol into a family of protocols with no name. The test "the defaults are the published protocol" is what prevents the defaults from drifting.</p></details>

<details><summary>8. Under the group's versioning rule, what version follows 1.4.2 after a fix that changes the curated output by 0.05 log units?</summary>
<p>2.0.0. The interface did not change, so semantic versioning in its usual reading would suggest 1.4.3, but any change that alters published numbers is treated as breaking: papers citing 1.4.2 and 2.0.0 report different results from the same input, and a reader must be able to see that from the version alone. The NEWS entry must say which numbers moved and by how much.</p></details>

<details><summary>9. What does a "software availability" statement commit the group to?</summary>
<p>That a repository exists at the stated address, that it contains the code that produced the published numbers (tagged or archived at a DOI, not just the current main branch), that it declares its dependencies and licence, and that it runs somewhere other than the author's laptop — which is what the <code>R-CMD-check</code> matrix demonstrates. Anything less turns the statement into a claim the group cannot support.</p></details>

<details><summary>10. Write three review comments on a pull request that adds a new filter to the protocol, in the group's style.</summary>
<p>"Blocking: this filter changes the compound count, so the golden test needs updating and NEWS.md needs a breaking-change entry — can you add both?"; "Could you move the 0.35 similarity cut-off into <code>curation_config()</code>? It is the kind of number a reviewer will ask about and it should have one definition."; "nit: <code>f</code> is hard to search for; <code>filter_by_similarity</code> would be easier to find in six months." The first is blocking and scientific, the second structural, the third a preference marked as such.</p></details>

<details><summary>11. Your package draws a figure as a side effect inside a computation function. Why is that a design error, and what is the fix?</summary>
<p>Because it mixes a pure computation with an output action: the function cannot be called in a loop, on a cluster or inside a test without opening a device or writing files, and its result can no longer be compared with an expected value. The fix is to return the numbers and let a separate plotting function — or the document — draw them. The same argument applies to functions that <code>print()</code> instead of returning, and to functions that use <code>message()</code> for results rather than for progress.</p></details>

<details><summary>12. A language model wrote a plausible implementation of a new aggregation rule. What do you do before merging it?</summary>
<p>Read it, then make it earn its place in the suite: add a hand-worked unit test for the new rule, check that the existing golden test either still passes or fails for a reason you can explain, run the static checks, and record the behavioural change in NEWS.md with a version bump. The tests are the contract; whether a human or a model wrote the code changes nothing about what must be demonstrated before it is trusted.</p></details>

## Glossary

<dl class="gloss">
<dt>Continuous integration (CI)</dt><dd>Automatic execution of <code>R CMD check</code> and the test suite on a clean machine on every push, across a matrix of operating systems and R versions.</dd>
<dt>Coverage</dt><dd>The fraction of code executed while the tests run; measured per line by <code>covr</code> and, here, per exported object by tracing the namespace. A diagnostic listing untested code, not a quality score.</dd>
<dt>DESCRIPTION</dt><dd>The metadata file that makes a directory an R package: name, version, licence, and the declared <code>Imports</code> and <code>Suggests</code>.</dd>
<dt>Fixture</dt><dd>A helper that builds the objects a test needs — in <code>testthat</code>, a function defined in a <code>helper-*.R</code> file, loaded automatically before the tests.</dd>
<dt>Golden-file (regression) test</dt><dd>A test comparing the output of a pipeline with a stored, reviewed reference; the guard that makes refactoring safe.</dd>
<dt>Idempotence</dt><dd>The property that applying an operation twice gives the same result as applying it once; here, re-aggregating an already aggregated table.</dd>
<dt>NAMESPACE</dt><dd>The generated file declaring which objects a package exports and which S3 methods it registers.</dd>
<dt>Pre-commit hook</dt><dd>A check that runs automatically before a commit is created, keeping failing code out of the history.</dd>
<dt>Property test</dt><dd>A test asserting an invariant that must hold for every input, rather than a specific output for a specific input.</dd>
<dt>Pure function</dt><dd>A function whose result depends only on its arguments and which changes nothing outside itself.</dd>
<dt>roxygen2</dt><dd>The package that turns <code>#'</code> comment blocks beside the code into <code>man/*.Rd</code> help pages and into <code>NAMESPACE</code>.</dd>
<dt>S3 class</dt><dd>R's lightest object system: an attribute naming the class, plus methods called <code>generic.class</code>; used here for the configuration and the log.</dd>
<dt>Semantic versioning</dt><dd>MAJOR.MINOR.PATCH numbering encoding compatibility; in this group extended so that any change to published numbers is a major release.</dd>
<dt>testthat</dt><dd>R's standard testing framework: <code>test_that()</code> blocks of <code>expect_*()</code> assertions, run by file, with helpers loaded automatically.</dd>
</dl>



In [ ]:
detach("package:rockcurate", unload = TRUE)
for (path in c(PKG, BUG, LIB, BUGLIB, file.path(TMP, "out"))) unlink(path, recursive = TRUE)
Sys.unsetenv(c("ROCKCURATE_RAW", "ROCKCURATE_REF", "ROCKCURATE_SCRIPT", "ROCKCURATE_LIB"))
cat("package sources removed:", !dir.exists(PKG), !dir.exists(BUG), "\n")
cat("temporary libraries removed:", !dir.exists(LIB), !dir.exists(BUGLIB), "\n")
cat("rockcurate still attached:", "package:rockcurate" %in% search(), "\n")



Nothing a document writes outside its own outputs should survive it. In a real project the package is a repository of its own, the document that uses it only calls `library(rockcurate)`, and the only thing the analysis keeps is the version number — which it records in the Methods, as Unit 23 required.

## References and further reading

<ol class="refs">
<li id="r1">Wilson G, Aruliah DA, Brown CT, et al. Best practices for scientific computing. <em>PLoS Biol</em>. 2014;12(1):e1001745. <a href="https://doi.org/10.1371/journal.pbio.1001745">doi:10.1371/journal.pbio.1001745</a></li>
<li id="r2">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <em>PLoS Comput Biol</em>. 2017;13(6):e1005510. <a href="https://doi.org/10.1371/journal.pcbi.1005510">doi:10.1371/journal.pcbi.1005510</a></li>
<li id="r3">Taschuk M, Wilson G. Ten simple rules for making research software more robust. <em>PLoS Comput Biol</em>. 2017;13(4):e1005412. <a href="https://doi.org/10.1371/journal.pcbi.1005412">doi:10.1371/journal.pcbi.1005412</a></li>
<li id="r4">Jimenez RC, Kuzak M, Alhamdoosh M, et al. Four simple recommendations to encourage best practices in research software. <em>F1000Research</em>. 2017;6:876. <a href="https://doi.org/10.12688/f1000research.11407.1">doi:10.12688/f1000research.11407.1</a></li>
<li id="r5">Smith AM, Katz DS, Niemeyer KE. Software citation principles. <em>PeerJ Comput Sci</em>. 2016;2:e86. <a href="https://doi.org/10.7717/peerj-cs.86">doi:10.7717/peerj-cs.86</a></li>
<li id="r6">Katz DS, Chue Hong NP, Clark T, et al. Recognizing the value of software: a software citation guide. <em>F1000Research</em>. 2021;9:1257. <a href="https://doi.org/10.12688/f1000research.26932.2">doi:10.12688/f1000research.26932.2</a></li>
<li id="r7">Chue Hong NP, Katz DS, Barker M, et al. Introducing the FAIR principles for research software. <em>Sci Data</em>. 2022;9:622. <a href="https://doi.org/10.1038/s41597-022-01710-x">doi:10.1038/s41597-022-01710-x</a></li>
<li id="r8">Prlic A, Procter JB. Ten simple rules for the open development of scientific software. <em>PLoS Comput Biol</em>. 2012;8(12):e1002802. <a href="https://doi.org/10.1371/journal.pcbi.1002802">doi:10.1371/journal.pcbi.1002802</a></li>
<li id="r9">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r10">Carver JC, Weber N, Ram K, Gesing S, Katz DS. A survey of the state of the practice for research software in the United States. <em>PeerJ Comput Sci</em>. 2022;8:e963. <a href="https://doi.org/10.7717/peerj-cs.963">doi:10.7717/peerj-cs.963</a></li>
<li id="r11">Perez-Riverol Y, Gatto L, Wang R, et al. Ten simple rules for taking advantage of Git and GitHub. <em>PLoS Comput Biol</em>. 2016;12(7):e1004947. <a href="https://doi.org/10.1371/journal.pcbi.1004947">doi:10.1371/journal.pcbi.1004947</a></li>
<li id="r12">Beaulieu-Jones BK, Greene CS. Reproducibility of computational workflows is automated using continuous analysis. <em>Nat Biotechnol</em>. 2017;35:342-346. <a href="https://doi.org/10.1038/nbt.3780">doi:10.1038/nbt.3780</a></li>
<li id="r13">Gruning B, Chilton J, Koster J, et al. Practical computational reproducibility in the life sciences. <em>Cell Syst</em>. 2018;6(6):631-635. <a href="https://doi.org/10.1016/j.cels.2018.03.014">doi:10.1016/j.cels.2018.03.014</a></li>
<li id="r14">Stodden V, Seiler J, Ma Z. An empirical analysis of journal policy effectiveness for computational reproducibility. <em>Proc Natl Acad Sci USA</em>. 2018;115(11):2584-2589. <a href="https://doi.org/10.1073/pnas.1708290115">doi:10.1073/pnas.1708290115</a></li>
<li id="r15">Perkel JM. Why Jupyter is data scientists' computational notebook of choice. <em>Nature</em>. 2018;563:145-146. <a href="https://doi.org/10.1038/d41586-018-07196-1">doi:10.1038/d41586-018-07196-1</a></li>
<li id="r16">Pimentel JF, Murta L, Braganholo V, Freire J. A large-scale study about quality and reproducibility of Jupyter notebooks. In: <em>Proc. 16th Int. Conf. on Mining Software Repositories (MSR)</em>; 2019:507-517. <a href="https://doi.org/10.1109/MSR.2019.00077">doi:10.1109/MSR.2019.00077</a></li>
<li id="r17">Rule A, Birmingham A, Zuniga C, et al. Ten simple rules for writing and sharing computational analyses in Jupyter notebooks. <em>PLoS Comput Biol</em>. 2019;15(7):e1007007. <a href="https://doi.org/10.1371/journal.pcbi.1007007">doi:10.1371/journal.pcbi.1007007</a></li>
<li id="r18">Baker M. 1,500 scientists lift the lid on reproducibility. <em>Nature</em>. 2016;533:452-454. <a href="https://doi.org/10.1038/533452a">doi:10.1038/533452a</a></li>
<li id="r19">Lee BD. Ten simple rules for documenting scientific software. <em>PLoS Comput Biol</em>. 2018;14(12):e1006561. <a href="https://doi.org/10.1371/journal.pcbi.1006561">doi:10.1371/journal.pcbi.1006561</a></li>
<li id="r20">Karimzadeh M, Hoffman MM. Top considerations for creating bioinformatics software documentation. <em>Brief Bioinform</em>. 2018;19(4):693-699. <a href="https://doi.org/10.1093/bib/bbw134">doi:10.1093/bib/bbw134</a></li>
<li id="r21">Kanewala U, Bieman JM. Testing scientific software: a systematic literature review. <em>Inf Softw Technol</em>. 2014;56(10):1219-1232. <a href="https://doi.org/10.1016/j.infsof.2014.05.006">doi:10.1016/j.infsof.2014.05.006</a></li>
<li id="r22">Sadowski C, Soderberg E, Church L, Sipko M, Bacchelli A. Modern code review: a case study at Google. In: <em>Proc. 40th Int. Conf. on Software Engineering: Software Engineering in Practice</em>; 2018:181-190. <a href="https://doi.org/10.1145/3183519.3183525">doi:10.1145/3183519.3183525</a></li>
<li id="r23">Petre M, Wilson G. Code review for and by scientists. <em>arXiv</em>. 2014:1407.5648. <a href="https://doi.org/10.48550/arXiv.1407.5648">doi:10.48550/arXiv.1407.5648</a></li>
<li id="r24">Miller G. A scientist's nightmare: software problem leads to five retractions. <em>Science</em>. 2006;314(5807):1856-1857. <a href="https://doi.org/10.1126/science.314.5807.1856">doi:10.1126/science.314.5807.1856</a></li>
<li id="r25">Merali Z. Computational science: error, why scientific programming does not compute. <em>Nature</em>. 2010;467:775-777. <a href="https://doi.org/10.1038/467775a">doi:10.1038/467775a</a></li>
<li id="r26">Eklund A, Nichols TE, Knutsson H. Cluster failure: why fMRI inferences for spatial extent have inflated false-positive rates. <em>Proc Natl Acad Sci USA</em>. 2016;113(28):7900-7905. <a href="https://doi.org/10.1073/pnas.1602413113">doi:10.1073/pnas.1602413113</a></li>
<li id="r27">Ziemann M, Eren Y, El-Osta A. Gene name errors are widespread in the scientific literature. <em>Genome Biol</em>. 2016;17:177. <a href="https://doi.org/10.1186/s13059-016-1044-7">doi:10.1186/s13059-016-1044-7</a></li>
<li id="r28">Hook D, Kelly D. Testing for trustworthiness in scientific software. In: <em>2009 ICSE Workshop on Software Engineering for Computational Science and Engineering</em>; 2009:59-64. <a href="https://doi.org/10.1109/SECSE.2009.5069163">doi:10.1109/SECSE.2009.5069163</a></li>
<li id="r29">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r30">Wickham H. testthat: get started with testing. <em>The R Journal</em>. 2011;3(1):5-10. <a href="https://doi.org/10.32614/RJ-2011-002">doi:10.32614/RJ-2011-002</a></li>
<li id="r31">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r32">Gruning B, Dale R, Sjodin A, et al. Bioconda: sustainable and comprehensive software distribution for the life sciences. <em>Nat Methods</em>. 2018;15:475-476. <a href="https://doi.org/10.1038/s41592-018-0046-7">doi:10.1038/s41592-018-0046-7</a></li>
<li id="r33">Wickham H, Averick M, Bryan J, et al. Welcome to the tidyverse. <em>J Open Source Softw</em>. 2019;4(43):1686. <a href="https://doi.org/10.21105/joss.01686">doi:10.21105/joss.01686</a></li>
<li id="r34">Xie Y. <em>Dynamic Documents with R and knitr</em>. 2nd ed. Chapman and Hall/CRC; 2015. <a href="https://doi.org/10.1201/9781315382487">doi:10.1201/9781315382487</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Wickham H, Bryan J. <em>R Packages</em>. 2nd ed. O'Reilly; 2023 (free online). - The reference for everything in Sections 3 to 12, written by the authors of the tooling.</li>
<li>Wickham H. <em>Advanced R</em>. 2nd ed. Chapman and Hall/CRC; 2019 (free online). - Environments, S3 and the copy-on-modify semantics that shape the design here.</li>
<li>Hunt A, Thomas D. <em>The Pragmatic Programmer</em>. 20th anniversary ed. Addison-Wesley; 2019. - The source of most of the habits in this unit, written for programmers of any domain.</li>
<li>Chacon S, Straub B. <em>Pro Git</em>. 2nd ed. Apress; 2014 (free online). - The version-control half of the workflow assumed here.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>testthat</code> documentation, especially the articles on test fixtures and on testing a package: <a href="https://testthat.r-lib.org/">testthat.r-lib.org</a>.</li>
<li>Writing R Extensions, the authority on package structure and on what <code>R CMD check</code> enforces: <a href="https://cran.r-project.org/doc/manuals/r-release/R-exts.html">cran.r-project.org — R-exts</a>.</li>
<li><code>roxygen2</code> and <code>pkgdown</code> documentation: <a href="https://roxygen2.r-lib.org/">roxygen2.r-lib.org</a>, <a href="https://pkgdown.r-lib.org/">pkgdown.r-lib.org</a>.</li>
<li><code>lintr</code>, <code>styler</code> and the R pre-commit hooks: <a href="https://lintr.r-lib.org/">lintr.r-lib.org</a>, <a href="https://styler.r-lib.org/">styler.r-lib.org</a>.</li>
<li>r-lib/actions, the GitHub Actions workflows for R packages used in Section 10: <a href="https://github.com/r-lib/actions">github.com/r-lib/actions</a>.</li>
<li>Citation File Format specification and the <code>cffr</code> package: <a href="https://citation-file-format.github.io/">citation-file-format.github.io</a>.</li>
<li>The Turing Way, chapters on research software engineering, testing and reviewing: <a href="https://the-turing-way.netlify.app/">the-turing-way.netlify.app</a>.</li>
</ul>
